# DeepVoice v3 Domain-Robust: 4 Pools × 25K Train/Validation + Independent OOD 2.5K + DACON Submit

이 노트북은 리더보드 분석 PDF의 진단을 반영한 새 학습·평가·제출 워크플로우입니다.

```text
4개 training source pool (각 6,250 = 총 25,000 source)
        │
        ├─ 22,500 DynamicMix train
        ├─  2,500 held-out validation
        │      ├─ clean view
        │      └─ domain-shift stress view
        │
        ├─ Presence branch: Log-Mel CNN, 2 heads
        └─ Fake branches
               ├─ AASIST, 3 heads
               └─ XLS-R 300M Dual-Graph, 3 heads
                        ↓ validation-only weight selection
                 Presence-gated Top-K aggregation

완전히 다른 4개 OOD pool (각 625 = 총 2,500 source)
        ↓ 고정 DACON형 DynamicMix 2,500 test
        ↓ File/Voice/Music EER + Voice/Music Presence AUC + Score

best checkpoints + fusion.json + offline code → submit.zip
```

핵심 변경점:

- Presence와 Fake detector를 분리해 5-head 간 gradient 충돌을 줄입니다.
- AASIST와 XLS-R Dual-Graph의 Fake 전용 3-head를 validation에서만 ensemble합니다.
- source-balanced sampling, 공통 codec/EQ/noise/loudness 처리와 clean sample 비율을 유지합니다.
- validation을 clean/stress 두 관점으로 평가해 동일 DynamicMix 규칙에만 맞는 모델을 피합니다.
- max 대신 Presence-gated Top-K mean으로 긴 파일의 단일 segment 오탐 영향을 줄입니다.
- OOD test는 학습·검증에 사용하지 않으며 결과를 보고 weight를 재조정하지 않습니다.

평가식과 제출 규격: [DACON 공식 평가 페이지](https://dacon.io/competitions/official/236749/overview/evaluation)


## 0. Colab 설치

GPU 런타임(T4/L4/A100)을 선택합니다. SONICS는 생성 없이 Hugging Face에서 두 개의 공식 ZIP을 내려받습니다. 학습 checkpoint와 manifest는 Drive에 저장되어 세션 재시작 후 이어집니다.


In [ ]:
import subprocess
import sys

packages = [
    "kaggle>=1.7", "transformers>=4.57,<5", "accelerate>=1.9", "huggingface_hub>=0.34",
    "librosa==0.10.2.post1", "soundfile>=0.12", "scikit-learn>=1.4",
    "panns-inference==0.1.1", "seaborn>=0.13", "pandas>=2.0",
    "scipy>=1.11", "einops>=0.8", "tqdm>=4.66",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("설치 완료. import 오류가 남으면 런타임을 한 번 재시작하세요.")


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path
from types import SimpleNamespace

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")
warnings.filterwarnings("ignore", category=FutureWarning)

CFG = SimpleNamespace(
    seed=42,
    sample_rate=16_000,
    # 공식 AASIST 입력 길이(64,600 samples @ 16 kHz)에 맞춘다.
    clip_seconds=4.0375,
    clip_samples=64_600,
    source_per_pool=6_250,
    source_split_counts={"train": 5_625, "validation": 625},
    recipe_counts={"train": 22_500, "validation": 2_500},
    panns_seconds=10,
    panns_batch=8,
    num_workers=4,
)
assert sum(CFG.source_split_counts.values()) == CFG.source_per_pool
assert sum(CFG.recipe_counts.values()) == 25_000

DACON_PROBABILITY_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
DACON_TRUTH_COLUMNS = [
    "FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT",
]
HEAD_WEIGHTS = torch.tensor([0.45, 0.18, 0.27, 0.05, 0.05], dtype=torch.float32)
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus", ".wma", ".amr"}

DRIVE_ROOT = Path("/content/drive/MyDrive/deepvoice_domainrobust_v3")
LOCAL_ROOT = Path("/content/deepvoice_domainrobust_v3")
VOICE_ROOT = LOCAL_ROOT / "voice_kaggle"
FMA_ROOT = LOCAL_ROOT / "fma"
SONICS_ROOT = LOCAL_ROOT / "sonics"
REPO_ROOT = LOCAL_ROOT / "repos"
RUN_ROOT = DRIVE_ROOT / "runs"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
for directory in (DRIVE_ROOT, LOCAL_ROOT, VOICE_ROOT, FMA_ROOT, SONICS_ROOT, REPO_ROOT, RUN_ROOT, MANIFEST_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.__version__)
print("source references:", f"{4 * CFG.source_per_pool:,}")
print("dynamic recipes:", f"{sum(CFG.recipe_counts.values()):,}")
print("free disk GB:", round(shutil.disk_usage("/content").free / 1024**3, 1))


## 1. Real/Fake Voice와 FMA 다운로드

Kaggle 데이터셋은 `real/` 9,066개, `fake/` 6,722개 구조이며 약 4.7GB입니다. FMA medium은 small+medium 30초 MP3 25,000개(약 22GiB)와 metadata를 공식 배포 URL에서 받습니다. FMA small만으로는 NoDerivatives를 올바르게 제외한 뒤 6,250곡이 남지 않으므로 medium 묶음을 사용합니다. 압축 파일 hash를 확인하고 재실행 시 기존 파일을 재사용합니다.


In [ ]:
KAGGLE_DATASET = "jayjoshi37/deepfake-audio-dataset-fake-vs-real-speech"
DOWNLOAD_VOICE = True
DOWNLOAD_FMA = True


def configure_kaggle_auth():
    from google.colab import files, userdata
    token = username = key = None
    try:
        token = userdata.get("KAGGLE_API_KEY")
    except Exception:
        pass
    if not token:
        try:
            username = userdata.get("KAGGLE_USERNAME")
            key = userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    elif username and key:
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key
    else:
        print("Kaggle Settings에서 받은 kaggle.json을 업로드하세요.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json이 업로드되지 않았습니다.")
        credential_dir = Path("/root/.kaggle")
        credential_dir.mkdir(parents=True, exist_ok=True)
        credential_path = credential_dir / "kaggle.json"
        credential_path.write_bytes(uploaded["kaggle.json"])
        credential_path.chmod(0o600)


def sha1_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha1()
    with Path(path).open("rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
if DOWNLOAD_VOICE and not voice_audio:
    configure_kaggle_auth()
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(VOICE_ROOT), "--unzip", "--quiet"],
        check=True,
    )

FMA_FILES = {
    "fma_medium.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_medium.zip",
        "c67b69ea232021025fca9231fc1c7c1a063ab50b",
    ),
    "fma_metadata.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        "f0df49ffe5f2a6008d7dc83c6915b31835dfe733",
    ),
}
if DOWNLOAD_FMA:
    for filename, (url, expected_sha1) in FMA_FILES.items():
        archive_path = FMA_ROOT / filename
        if not archive_path.exists():
            subprocess.run(["wget", "-q", "--show-progress", "-O", str(archive_path), url], check=True)
        actual_sha1 = sha1_file(archive_path)
        if actual_sha1 != expected_sha1:
            raise RuntimeError(f"FMA hash mismatch: {filename} {actual_sha1}")
        marker = FMA_ROOT / (filename + ".extracted")
        if not marker.exists():
            subprocess.run(["unzip", "-q", "-o", str(archive_path), "-d", str(FMA_ROOT)], check=True)
            marker.write_text(actual_sha1, encoding="utf-8")

voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
fma_audio = [p for p in (FMA_ROOT / "fma_medium").rglob("*.mp3") if p.is_file()]
print("voice audio:", f"{len(voice_audio):,}")
print("FMA medium audio:", f"{len(fma_audio):,}")
if len(voice_audio) < 2 * CFG.source_per_pool:
    raise RuntimeError("Kaggle real/fake 음성 파일이 충분하지 않습니다.")
if len(fma_audio) < CFG.source_per_pool:
    raise RuntimeError("FMA real music 파일이 충분하지 않습니다.")


## 2. SONICS fake music 다운로드

[SONICS 공식 데이터셋](https://huggingface.co/datasets/awsaf49/sonics)은 49,074개의 Suno/Udio 가짜 곡을 10개 ZIP으로 제공합니다. 각 ZIP에는 5,000곡이 들어 있으므로 `part_01.zip`과 `part_02.zip`만 내려받아 10,000곡 중 6,250곡을 선택합니다. 전체 32.6GB 저장소를 받을 필요가 없습니다.

SONICS는 가짜 곡 오디오만 제공하므로 real music은 FMA를 계속 사용합니다. 데이터셋 라이선스는 CC BY-NC 4.0이며, metadata의 `source`, `algorithm`, `label`, `split`, `no_vocal`을 source manifest에 보존합니다.


In [ ]:
from huggingface_hub import hf_hub_download

SONICS_REPO = "awsaf49/sonics"
SONICS_REVISION = "3788dca9f9f11ad92e9097ef4b58eee247661e7f"
SONICS_PARTS = ["fake_songs/part_01.zip", "fake_songs/part_02.zip"]
DOWNLOAD_SONICS = True
DELETE_SONICS_ARCHIVES_AFTER_EXTRACT = True
SONICS_METADATA_PATH = SONICS_ROOT / "fake_songs.csv"
SONICS_AUDIO_ROOT = SONICS_ROOT / "fake_songs"
SONICS_ROOT.mkdir(parents=True, exist_ok=True)


def download_sonics_file(filename):
    return Path(hf_hub_download(
        repo_id=SONICS_REPO,
        repo_type="dataset",
        filename=filename,
        revision=SONICS_REVISION,
        local_dir=str(SONICS_ROOT),
    ))


if DOWNLOAD_SONICS:
    if not SONICS_METADATA_PATH.exists():
        downloaded_metadata = download_sonics_file("fake_songs.csv")
        if downloaded_metadata.resolve() != SONICS_METADATA_PATH.resolve():
            shutil.copy2(downloaded_metadata, SONICS_METADATA_PATH)

    for part_name in SONICS_PARTS:
        part_stem = Path(part_name).stem
        marker = SONICS_ROOT / f".{part_stem}.extracted"
        if marker.exists():
            print("reuse extracted:", part_name)
            continue
        archive_path = download_sonics_file(part_name)
        with zipfile.ZipFile(archive_path) as archive:
            members = [info for info in archive.infolist() if not info.is_dir()]
            if len(members) != 5_000:
                raise RuntimeError(f"SONICS {part_name}: expected 5,000 files, got {len(members):,}")
            archive.extractall(SONICS_ROOT)
        marker.write_text(f"{SONICS_REVISION}\n{len(members)} files\n", encoding="utf-8")
        if DELETE_SONICS_ARCHIVES_AFTER_EXTRACT and archive_path.exists():
            archive_path.unlink()
elif not SONICS_METADATA_PATH.exists() or not SONICS_AUDIO_ROOT.exists():
    raise FileNotFoundError("DOWNLOAD_SONICS=False이면 SONICS metadata와 fake_songs 폴더를 직접 준비해야 합니다.")

sonics_audio = sorted(
    path for path in SONICS_AUDIO_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES and path.stat().st_size > 0
)
sonics_metadata = pd.read_csv(
    SONICS_METADATA_PATH,
    usecols=["filename", "algorithm", "source", "label", "target", "skip_time", "no_vocal", "split"],
    low_memory=False,
)
sonics_metadata = sonics_metadata[sonics_metadata["target"].eq(1)].copy()
sonics_metadata["file_stem"] = sonics_metadata["filename"].astype(str).map(lambda value: Path(value).stem)
if sonics_metadata["no_vocal"].dtype != bool:
    sonics_metadata["no_vocal"] = (
        sonics_metadata["no_vocal"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    )
sonics_metadata = sonics_metadata.drop_duplicates("file_stem", keep="last")
sonics_metadata_lookup = sonics_metadata.set_index("file_stem").to_dict("index")
sonics_candidates = [
    str(path.resolve()) for path in sonics_audio if path.stem in sonics_metadata_lookup
]
print({
    "SONICS extracted audio": len(sonics_audio),
    "metadata fake rows": len(sonics_metadata),
    "matched candidates": len(sonics_candidates),
    "sources": sonics_metadata.loc[
        sonics_metadata["file_stem"].isin({Path(path).stem for path in sonics_candidates}), "source"
    ].value_counts().to_dict(),
})
if len(sonics_candidates) < CFG.source_per_pool:
    raise RuntimeError(f"SONICS fake song이 {CFG.source_per_pool:,}개 필요합니다.")


## 3. 네 소스 풀에서 각각 정확히 6,250개 선택

Kaggle 폴더의 `real/`, `fake/`를 음성 라벨로 사용합니다. FMA는 metadata에서 small+medium subset을 확인하고 `NoDerivatives` 계열을 제외합니다. SONICS는 다운로드한 10,000곡 중 metadata와 일치하는 가짜 곡을 선택합니다. 모든 선택은 stable hash 정렬을 사용해 재실행해도 동일합니다.


In [ ]:
def stable_int(text):
    return int(hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16], 16)


def select_exact(paths, count, namespace):
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    ordered = sorted(unique_paths, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{value}"))
    if len(ordered) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(ordered):,}개 발견")
    return ordered[:count]


def select_balanced_exact(paths, count, namespace, group_lookup):
    """그룹별 round-robin으로 선택해 한 장르/생성기가 풀을 지배하지 않게 한다."""
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    if len(unique_paths) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(unique_paths):,}개 발견")
    queues = {}
    for path in unique_paths:
        group = str(group_lookup.get(path, "unknown") or "unknown")
        queues.setdefault(group, []).append(path)
    for group, values in queues.items():
        queues[group] = sorted(
            values, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{group}|{value}")
        )
    groups = sorted(queues, key=lambda group: stable_int(f"group|{CFG.seed}|{namespace}|{group}"))
    selected_paths = []
    cursor = 0
    while len(selected_paths) < count:
        progressed = False
        for group in groups:
            if cursor < len(queues[group]):
                selected_paths.append(queues[group][cursor])
                progressed = True
                if len(selected_paths) == count:
                    break
        if not progressed:
            break
        cursor += 1
    if len(selected_paths) != count:
        raise RuntimeError(f"{namespace}: balanced selection failed ({len(selected_paths):,}/{count:,})")
    return selected_paths


real_voice_all, fake_voice_all, unresolved_voice = [], [], []
for path in voice_audio:
    parts = {part.lower() for part in path.parts}
    if "real" in parts and "fake" not in parts:
        real_voice_all.append(path)
    elif "fake" in parts and "real" not in parts:
        fake_voice_all.append(path)
    else:
        unresolved_voice.append(path)
print({"real_voice": len(real_voice_all), "fake_voice": len(fake_voice_all), "unresolved": len(unresolved_voice)})

tracks_path = FMA_ROOT / "fma_metadata" / "tracks.csv"
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1], low_memory=False)
small_medium_tracks = tracks[
    tracks[("set", "subset")].astype(str).isin({"small", "medium"})
].copy()

def fma_track_path(track_id):
    track_id = int(track_id)
    return FMA_ROOT / "fma_medium" / f"{track_id:06d}"[:3] / f"{track_id:06d}.mp3"

def fma_license_allowed(value):
    value = str(value).lower()
    if not value or value == "nan":
        return False
    compact = re.sub(r"[^a-z0-9]+", "", value)
    normalized = re.sub(r"[^a-z0-9]+", " ", value)
    no_derivatives = (
        "noderivative" in compact
        or "noderivs" in compact
        or "musicsharing" in compact
        or re.search(r"\bby\s+(?:nc\s+)?nd\b", normalized) is not None
    )
    return not no_derivatives

fma_candidates = []
fma_license_lookup = {}
fma_genre_lookup = {}
fma_artist_lookup = {}
for track_id, row in small_medium_tracks.iterrows():
    path = fma_track_path(track_id)
    license_value = row.get(("track", "license"), "")
    if path.is_file() and path.stat().st_size > 0 and fma_license_allowed(license_value):
        resolved_path = str(path.resolve())
        fma_candidates.append(path)
        fma_license_lookup[resolved_path] = str(license_value)
        genre_value = str(row.get(("track", "genre_top"), "unknown")).strip()
        fma_genre_lookup[resolved_path] = genre_value if genre_value and genre_value != "nan" else "unknown"
        artist_value = str(row.get(("artist", "id"), "unknown")).strip()
        fma_artist_lookup[resolved_path] = artist_value if artist_value and artist_value != "nan" else "unknown"

sonics_balance_lookup = {}
for path in sonics_candidates:
    record = sonics_metadata_lookup.get(Path(path).stem, {})
    source_name = str(record.get("source", "unknown") or "unknown")
    algorithm_name = str(record.get("algorithm", "unknown") or "unknown")
    sonics_balance_lookup[str(Path(path).resolve())] = f"{source_name}|{algorithm_name}"

selected = {
    "real_voice": select_exact(real_voice_all, CFG.source_per_pool, "real_voice"),
    "fake_voice": select_exact(fake_voice_all, CFG.source_per_pool, "fake_voice"),
    "real_music": select_balanced_exact(
        fma_candidates, CFG.source_per_pool, "real_music", fma_genre_lookup
    ),
    "fake_music": select_balanced_exact(
        sonics_candidates, CFG.source_per_pool, "fake_music", sonics_balance_lookup
    ),
}
assert sum(map(len, selected.values())) == 25_000
if set(selected["real_voice"]) & set(selected["fake_voice"]):
    raise RuntimeError("real/fake voice pool overlap")
display(pd.DataFrame({name: [len(paths)] for name, paths in selected.items()}))
print("FMA eligible after license filter:", f"{len(fma_candidates):,}")
print("balanced FMA genres:", pd.Series([fma_genre_lookup[path] for path in selected["real_music"]]).value_counts().head(20).to_dict())
print("balanced SONICS groups:", pd.Series([sonics_balance_lookup[path] for path in selected["fake_music"]]).value_counts().to_dict())


### 3.1 화자·아티스트·곡 group-disjoint 재선택

정확히 6,250개를 먼저 임의 분할하지 않고 전체 후보에서 validation group을 먼저 격리한 뒤 625개를 뽑고, 나머지 group에서 train 5,625개를 뽑습니다. Voice는 파일명 화자명이 유효하면 그 값을 쓰고, 숫자 파일명처럼 화자 metadata가 없으면 MFCC pseudo-speaker 64군집을 Drive에 캐시합니다. FMA는 `artist_id`, SONICS는 곡 파일 ID를 group으로 사용합니다. 아래 검사가 같은 group의 train/validation 중복을 차단합니다.


In [ ]:
def infer_voice_speaker(path):
    stem = Path(path).stem.lower()
    stem = re.sub(r"(?:[_-](?:chunk|segment|part|clip))?[_-]?\d+$", "", stem)
    converted = re.split(r"(?:-to-|_to_|\s+to\s+)", stem)
    identity = converted[-1]
    identity = re.sub(r"(?:[_-](?:real|fake|original|converted))+$", "", identity)
    identity = re.sub(r"[^a-z0-9가-힣]+", "_", identity).strip("_")
    return identity or Path(path).stem.lower()


def make_group_lookup(paths, resolver, namespace):
    lookup = {str(Path(path).resolve()): str(resolver(str(Path(path).resolve()))) for path in paths}
    counts = pd.Series(lookup.values()).value_counts()
    if len(counts) < 2:
        warnings.warn(f"{namespace}: 화자 group을 파일명에서 복원하지 못해 파일 단위 group으로 대체합니다.")
        lookup = {path: f"file:{Path(path).stem}" for path in lookup}
    return lookup


VOICE_FEATURE_CACHE = MANIFEST_ROOT / "voice_pseudo_speaker_features.csv"
VOICE_GROUP_CACHE = MANIFEST_ROOT / "voice_pseudo_speaker_groups.csv"


def speaker_feature(path):
    audio, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, duration=4.0)
    if len(audio) < 2_048:
        raise ValueError("too short")
    audio = librosa.util.normalize(np.asarray(audio, np.float32))
    mfcc = librosa.feature.mfcc(y=audio, sr=CFG.sample_rate, n_mfcc=24, n_fft=512, hop_length=160)
    delta = librosa.feature.delta(mfcc, width=min(9, mfcc.shape[1] // 2 * 2 - 1)) if mfcc.shape[1] >= 5 else np.zeros_like(mfcc)
    stacked = np.concatenate([mfcc, delta], axis=0)
    return np.concatenate([stacked.mean(1), stacked.std(1)]).astype(np.float32)


def pseudo_speaker_group_lookups(real_paths, fake_paths, clusters=64):
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler

    all_paths = sorted({str(Path(path).resolve()) for path in [*real_paths, *fake_paths]})
    if VOICE_FEATURE_CACHE.exists():
        feature_frame = pd.read_csv(VOICE_FEATURE_CACHE)
    else:
        feature_frame = pd.DataFrame(columns=["path", *[f"f{i:03d}" for i in range(96)]])
    feature_frame = feature_frame.drop_duplicates("path", keep="last")
    completed = set(feature_frame.path.astype(str))
    rows = feature_frame.to_dict("records")
    for index, path in enumerate(tqdm([p for p in all_paths if p not in completed], desc="voice pseudo-speaker features"), 1):
        try:
            feature = speaker_feature(path)
            rows.append({"path": path, **{f"f{i:03d}": float(value) for i, value in enumerate(feature)}})
        except Exception as exc:
            print("speaker feature skip:", path, repr(exc))
        if index % 128 == 0:
            pd.DataFrame(rows).to_csv(VOICE_FEATURE_CACHE, index=False)
    feature_frame = pd.DataFrame(rows).drop_duplicates("path", keep="last")
    feature_frame.to_csv(VOICE_FEATURE_CACHE, index=False)
    feature_columns = [column for column in feature_frame if column.startswith("f")]
    if len(feature_frame) < 2 * CFG.source_per_pool or len(feature_columns) != 96:
        raise RuntimeError(f"pseudo-speaker feature shortage: {len(feature_frame):,}, dims={len(feature_columns)}")
    matrix = StandardScaler().fit_transform(feature_frame[feature_columns].to_numpy(np.float32))
    labels = MiniBatchKMeans(
        n_clusters=min(clusters, len(feature_frame) // 50), random_state=CFG.seed,
        batch_size=1024, n_init=10,
    ).fit_predict(matrix)
    grouped = pd.DataFrame({"path": feature_frame.path.astype(str), "split_group": [f"pseudo_{x:03d}" for x in labels]})
    grouped.to_csv(VOICE_GROUP_CACHE, index=False)
    lookup = dict(zip(grouped.path, grouped.split_group))
    return lookup


def build_voice_group_lookups(real_paths, fake_paths):
    parsed_real = make_group_lookup(real_paths, infer_voice_speaker, "real_voice")
    parsed_fake = make_group_lookup(fake_paths, infer_voice_speaker, "fake_voice")
    real_counts = pd.Series(parsed_real.values()).value_counts()
    fake_counts = pd.Series(parsed_fake.values()).value_counts()
    common = set(real_counts.index) & set(fake_counts.index)
    metadata_is_usable = (
        len(common) >= 4 and real_counts.median() >= 3 and fake_counts.median() >= 3
    )
    if metadata_is_usable:
        print("voice split groups: filename speaker metadata")
        return parsed_real, parsed_fake
    print("voice filenames have no usable speaker ID; building cached MFCC pseudo-speaker groups")
    combined = pseudo_speaker_group_lookups(real_paths, fake_paths)
    return (
        {str(Path(path).resolve()): combined[str(Path(path).resolve())] for path in real_paths if str(Path(path).resolve()) in combined},
        {str(Path(path).resolve()): combined[str(Path(path).resolve())] for path in fake_paths if str(Path(path).resolve()) in combined},
    )


def select_group_disjoint(paths, counts, namespace, group_lookup, balance_lookup=None):
    paths = sorted({str(Path(path).resolve()) for path in paths})
    train_count, validation_count = counts["train"], counts["validation"]
    groups = {}
    for path in paths:
        groups.setdefault(str(group_lookup[path]), []).append(path)
    ordered_groups = sorted(groups, key=lambda group: stable_int(f"holdout|{CFG.seed}|{namespace}|{group}"))
    validation_groups, heldout_size = [], 0
    for group in ordered_groups:
        if len(paths) - heldout_size - len(groups[group]) < train_count:
            continue
        validation_groups.append(group); heldout_size += len(groups[group])
        if heldout_size >= validation_count:
            break
    if heldout_size < validation_count:
        raise RuntimeError(f"{namespace}: group-disjoint validation {validation_count}개를 구성할 수 없습니다.")
    heldout = [path for group in validation_groups for path in groups[group]]
    train_candidates = [path for group, values in groups.items() if group not in validation_groups for path in values]
    selector = select_balanced_exact if balance_lookup is not None else select_exact
    if balance_lookup is None:
        validation = selector(heldout, validation_count, namespace + "_validation")
        train = selector(train_candidates, train_count, namespace + "_train")
    else:
        validation = selector(heldout, validation_count, namespace + "_validation", balance_lookup)
        train = selector(train_candidates, train_count, namespace + "_train", balance_lookup)
    split = {path: "train" for path in train}; split.update({path: "validation" for path in validation})
    selected_paths = train + validation
    selected_groups = {path: group_lookup[path] for path in selected_paths}
    train_groups = {selected_groups[path] for path in train}; val_groups = {selected_groups[path] for path in validation}
    if train_groups & val_groups:
        raise RuntimeError(f"{namespace}: group leakage")
    return selected_paths, split, selected_groups


def select_joint_voice_group_disjoint(pool_paths, pool_group_lookup, counts):
    normalized = {
        pool: sorted({str(Path(path).resolve()) for path in paths})
        for pool, paths in pool_paths.items()
    }
    grouped = {}
    for pool, paths in normalized.items():
        by_group = {}
        for path in paths:
            by_group.setdefault(pool_group_lookup[pool][path], []).append(path)
        grouped[pool] = by_group
    common_groups = set.intersection(*(set(value) for value in grouped.values()))
    train_count, validation_count = counts["train"], counts["validation"]
    validation_groups, heldout = [], {pool: 0 for pool in normalized}
    for group in sorted(common_groups, key=lambda value: stable_int(f"voice-holdout|{CFG.seed}|{value}")):
        proposed = {pool: heldout[pool] + len(grouped[pool][group]) for pool in normalized}
        if any(len(normalized[pool]) - proposed[pool] < train_count for pool in normalized):
            continue
        validation_groups.append(group); heldout = proposed
        if all(value >= validation_count for value in heldout.values()):
            break
    if not validation_groups or not all(value >= validation_count for value in heldout.values()):
        raise RuntimeError(
            "real/fake voice 공통 화자 holdout을 만들지 못했습니다. "
            "infer_voice_speaker()를 실제 파일명 규칙에 맞게 조정하세요."
        )
    selected, splits, chosen_groups = {}, {}, {}
    for pool, paths in normalized.items():
        validation_candidates = [path for group in validation_groups for path in grouped[pool][group]]
        train_candidates = [path for group, values in grouped[pool].items() if group not in validation_groups for path in values]
        validation = select_exact(validation_candidates, validation_count, pool + "_validation")
        train = select_exact(train_candidates, train_count, pool + "_train")
        selected[pool] = train + validation
        splits[pool] = {path: "train" for path in train} | {path: "validation" for path in validation}
        chosen_groups[pool] = {path: pool_group_lookup[pool][path] for path in selected[pool]}
    return selected, splits, chosen_groups


voice_real_groups, voice_fake_groups = build_voice_group_lookups(real_voice_all, fake_voice_all)
real_voice_all = [path for path in real_voice_all if str(Path(path).resolve()) in voice_real_groups]
fake_voice_all = [path for path in fake_voice_all if str(Path(path).resolve()) in voice_fake_groups]
fma_groups = {str(Path(path).resolve()): fma_artist_lookup[str(Path(path).resolve())] for path in fma_candidates}
sonics_song_groups = {str(Path(path).resolve()): Path(path).stem for path in sonics_candidates}

selected, preassigned_split, selection_group_lookup = select_joint_voice_group_disjoint(
    {"real_voice": real_voice_all, "fake_voice": fake_voice_all},
    {"real_voice": voice_real_groups, "fake_voice": voice_fake_groups},
    CFG.source_split_counts,
)
for pool, paths, group_lookup, balance_lookup in (
    ("real_music", fma_candidates, fma_groups, fma_genre_lookup),
    ("fake_music", sonics_candidates, sonics_song_groups, sonics_balance_lookup),
):
    chosen, split, chosen_groups = select_group_disjoint(
        paths, CFG.source_split_counts, pool, group_lookup, balance_lookup
    )
    selected[pool] = chosen
    preassigned_split[pool] = split
    selection_group_lookup[pool] = chosen_groups
assert sum(map(len, selected.values())) == 25_000
display(pd.DataFrame({pool: pd.Series(split).value_counts() for pool, split in preassigned_split.items()}).fillna(0).astype(int))


## 4. FMA 보컬 스크리닝

PANNs Cnn14의 AudioSet `Speech`, `Singing`, `Choir`, `Vocal music` 계열 점수로 FMA track의 보컬 가능성을 기록합니다. 모델이 music-only로 잘못 학습하지 않도록, 검출된 track은 real music과 real voice가 함께 존재하는 source로 취급합니다.

스크리닝 결과는 Drive CSV에 누적 저장되어 중단 후 이어집니다. `RUN_FMA_VOCAL_SCREEN=False`는 빠른 디버그에만 사용하세요.


In [ ]:
RUN_FMA_VOCAL_SCREEN = True
FMA_VOCAL_THRESHOLD = 0.20
FMA_SCREEN_PATH = MANIFEST_ROOT / "fma_panns_vocal_screen.csv"

if FMA_SCREEN_PATH.exists():
    fma_screen = pd.read_csv(FMA_SCREEN_PATH)
else:
    fma_screen = pd.DataFrame(columns=["path", "vocal_score", "music_score", "contains_voice", "screen_ok"])

def as_boolean(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

if RUN_FMA_VOCAL_SCREEN:
    from panns_inference import AudioTagging, labels as panns_labels

    vocal_names = [
        "Speech", "Male speech, man speaking", "Female speech, woman speaking",
        "Conversation", "Narration, monologue", "Singing", "Choir", "Vocal music",
    ]
    vocal_indices = [panns_labels.index(name) for name in vocal_names if name in panns_labels]
    music_index = panns_labels.index("Music")
    completed = set(fma_screen.loc[as_boolean(fma_screen["screen_ok"]), "path"].astype(str)) if len(fma_screen) else set()
    missing = [path for path in selected["real_music"] if path not in completed]

    def load_for_panns(path):
        audio, _ = librosa.load(path, sr=32_000, mono=True, duration=CFG.panns_seconds)
        target = 32_000 * CFG.panns_seconds
        if len(audio) < target:
            audio = np.pad(audio, (0, target - len(audio)))
        return np.asarray(audio[:target], dtype=np.float32)

    new_rows = []
    if missing:
        checkpoint_path = DRIVE_ROOT / "panns" / "Cnn14_mAP=0.431.pth"
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        tagger = AudioTagging(checkpoint_path=str(checkpoint_path), device=str(DEVICE))
    for offset in tqdm(range(0, len(missing), CFG.panns_batch), desc="FMA PANNs vocal screen"):
        batch_paths = missing[offset:offset + CFG.panns_batch]
        batch_audio, ok_flags = [], []
        for path in batch_paths:
            try:
                batch_audio.append(load_for_panns(path))
                ok_flags.append(True)
            except Exception:
                batch_audio.append(np.zeros(32_000 * CFG.panns_seconds, dtype=np.float32))
                ok_flags.append(False)
        clipwise, _ = tagger.inference(np.stack(batch_audio))
        for path, scores, ok in zip(batch_paths, clipwise, ok_flags):
            vocal_score = float(np.max(scores[vocal_indices])) if vocal_indices else 0.0
            new_rows.append({
                "path": path, "vocal_score": vocal_score,
                "music_score": float(scores[music_index]),
                "contains_voice": bool(vocal_score >= FMA_VOCAL_THRESHOLD),
                "screen_ok": bool(ok),
            })
        if len(new_rows) >= 128 or offset + CFG.panns_batch >= len(missing):
            fma_screen = pd.concat([fma_screen, pd.DataFrame(new_rows)], ignore_index=True)
            fma_screen = fma_screen.drop_duplicates("path", keep="last")
            fma_screen.to_csv(FMA_SCREEN_PATH, index=False, encoding="utf-8")
            new_rows = []
    if missing:
        del tagger
    gc.collect()
    torch.cuda.empty_cache()
else:
    fma_screen = pd.DataFrame({
        "path": selected["real_music"], "vocal_score": 0.0, "music_score": np.nan,
        "contains_voice": False, "screen_ok": False,
    })
    print("경고: FMA 보컬 스크리닝을 끈 상태입니다. music-only 라벨 노이즈가 생길 수 있습니다.")

fma_screen = fma_screen[fma_screen["path"].isin(selected["real_music"])].copy()
fma_screen["contains_voice"] = as_boolean(fma_screen["contains_voice"])
fma_screen["screen_ok"] = as_boolean(fma_screen["screen_ok"])
print("FMA screened:", len(fma_screen), "vocal-like:", int(fma_screen["contains_voice"].astype(bool).sum()))
display(fma_screen.describe(include="all").T)


## 5. Source manifest와 누수 없는 split

각 풀의 6,250개를 5,625/625로 정확히 나눕니다. 동일 source path가 split을 넘지 않는지 검사하고, 데이터 출처·license·FMA 보컬 스크리닝 값을 manifest에 보존합니다.


In [ ]:
fma_voice_lookup = dict(zip(fma_screen["path"].astype(str), fma_screen["contains_voice"].astype(bool)))
rows = []
for pool, paths in selected.items():
    boundaries = {
        split_name: [path for path in paths if preassigned_split[pool][path] == split_name]
        for split_name in CFG.source_split_counts
    }
    assert {name: len(values) for name, values in boundaries.items()} == CFG.source_split_counts
    for split_name, split_paths in boundaries.items():
        for path in split_paths:
            sonics_record = sonics_metadata_lookup.get(Path(path).stem, {}) if pool == "fake_music" else {}
            if pool == "real_music":
                contains_voice = bool(fma_voice_lookup.get(path, False))
            elif pool == "fake_music":
                contains_voice = not bool(sonics_record.get("no_vocal", False))
            else:
                contains_voice = pool.endswith("voice")
            rows.append({
                "source_id": hashlib.sha256(f"{pool}|{path}".encode()).hexdigest()[:20],
                "pool": pool, "split": split_name, "path": path,
                "split_group": selection_group_lookup[pool][path],
                "contains_voice": contains_voice,
                "voice_fake": int(pool == "fake_voice" or (pool == "fake_music" and contains_voice)),
                "contains_music": pool.endswith("music"),
                "music_fake": int(pool == "fake_music"),
                "license": (
                    "CC-BY-SA-4.0" if pool.endswith("voice") else
                    fma_license_lookup.get(path, "FMA artist-selected") if pool == "real_music" else
                    "CC-BY-NC-4.0"
                ),
                "origin": (
                    KAGGLE_DATASET if pool.endswith("voice") else
                    "FMA medium" if pool == "real_music" else SONICS_REPO
                ),
                "source_detail": str(sonics_record.get("source", "")),
                "generation_algorithm": str(sonics_record.get("algorithm", "")),
                "sonics_label": str(sonics_record.get("label", "")),
                "sonics_original_split": str(sonics_record.get("split", "")),
                "genre_or_style": (
                    fma_genre_lookup.get(path, "unknown") if pool == "real_music" else
                    sonics_balance_lookup.get(path, "unknown") if pool == "fake_music" else
                    "speech"
                ),
                "balance_group": (
                    fma_genre_lookup.get(path, "unknown") if pool == "real_music" else
                    sonics_balance_lookup.get(path, "unknown") if pool == "fake_music" else
                    "speech"
                ),
                "artist_group": fma_artist_lookup.get(path, "") if pool == "real_music" else "",
                "original_suffix": Path(path).suffix.lower(),
            })
source_manifest = pd.DataFrame(rows)
assert len(source_manifest) == 25_000
assert source_manifest["source_id"].is_unique
assert source_manifest.groupby("pool").size().eq(CFG.source_per_pool).all()
assert source_manifest.groupby(["pool", "split"]).size().to_dict() == {
    (pool, split_name): count
    for pool in selected for split_name, count in CFG.source_split_counts.items()
}
if source_manifest.groupby("path")["split"].nunique().max() != 1:
    raise RuntimeError("source path가 여러 split에 존재합니다.")
SOURCE_MANIFEST_PATH = MANIFEST_ROOT / "source_manifest_25000.csv"
source_manifest.to_csv(SOURCE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(source_manifest["pool"], source_manifest["split"], margins=True))
display(source_manifest.groupby(["pool", "license"]).size().rename("count").reset_index())
display(source_manifest[source_manifest["pool"].eq("fake_music")][
    ["source_detail", "generation_algorithm", "sonics_label", "contains_voice"]
].value_counts().rename("count").reset_index().head(30))
display(source_manifest.groupby(["pool", "balance_group"]).size().rename("count").reset_index())
display(pd.crosstab(source_manifest["pool"], source_manifest["original_suffix"], margins=True))
print("saved:", SOURCE_MANIFEST_PATH)


In [ ]:
group_leakage = (
    source_manifest.groupby(["pool", "split_group"])["split"].nunique().gt(1)
)
if group_leakage.any():
    raise RuntimeError("speaker/artist/song group leakage detected")
voice_group_leakage = (
    source_manifest[source_manifest.pool.str.endswith("voice")]
    .groupby("split_group")["split"].nunique().gt(1)
)
if voice_group_leakage.any():
    raise RuntimeError("real/fake voice 사이의 speaker group leakage detected")
display(source_manifest.groupby(["pool", "split"])["split_group"].nunique().unstack(fill_value=0))
sonics_split = source_manifest[source_manifest.pool.eq("fake_music")].copy()
display(pd.crosstab(sonics_split["source_detail"], sonics_split["split"], margins=True))
print("speaker/artist/song group leakage: 0")


## 6. 정확히 25,000개 Dynamic Mix recipe

여덟 조합을 균형 배치합니다: RV, FV, RM, FM, RV+RM, FV+RM, RV+FM, FV+FM. Recipe CSV에는 실제 waveform 대신 split, 조합, seed를 저장합니다. Train은 `epoch`을 seed에 포함해 매 epoch 다른 source/crop/SNR/layout을 만들고 validation은 고정합니다.


In [ ]:
RECIPE_TYPES = ["rv", "fv", "rm", "fm", "rv_rm", "fv_rm", "rv_fm", "fv_fm"]


def build_recipe_manifest():
    recipe_rows = []
    for split_name, count in CFG.recipe_counts.items():
        recipe_types = [RECIPE_TYPES[index % len(RECIPE_TYPES)] for index in range(count)]
        random.Random(CFG.seed + stable_int(split_name)).shuffle(recipe_types)
        for index, recipe_type in enumerate(recipe_types):
            recipe_rows.append({
                "recipe_id": f"{split_name}_{index:05d}",
                "split": split_name, "recipe_type": recipe_type,
                "seed": stable_int(f"recipe|{CFG.seed}|{split_name}|{index}|{recipe_type}") % (2**31 - 1),
            })
    return pd.DataFrame(recipe_rows)


recipe_manifest = build_recipe_manifest()
assert len(recipe_manifest) == 25_000
assert recipe_manifest["recipe_id"].is_unique
assert recipe_manifest.groupby("split").size().to_dict() == CFG.recipe_counts
RECIPE_MANIFEST_PATH = MANIFEST_ROOT / "train_validation_recipes_25000.csv"
recipe_manifest.to_csv(RECIPE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(recipe_manifest["recipe_type"], recipe_manifest["split"], margins=True))
print("saved:", RECIPE_MANIFEST_PATH)


## 7. 공식 AASIST·RawBoost 소스와 오디오 로더

SoundFile → librosa → FFmpeg 순서로 WAV/MP3/FLAC 등을 처리합니다. 긴 파일은 random crop, 짧은 파일은 반복 후 crop합니다. RawBoost는 train 최종 mixture에만 적용합니다.


In [ ]:
repositories = {
    "aasist": ("https://github.com/clovaai/aasist.git", "a04c9863f63d44471dde8a6abcb3b082b07cd1d1"),
    "rawboost": ("https://github.com/TakHemlata/RawBoost-antispoofing.git", "4f161a8b4d0d9f4a8431509ddd86c645869ef6c4"),
}
repo_commits = {}
for name, (url, commit) in repositories.items():
    destination = REPO_ROOT / name
    if not destination.exists():
        subprocess.run(["git", "clone", "--no-checkout", url, str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "fetch", "--depth", "1", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "--detach", commit], check=True)
    repo_commits[name] = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True,
    ).strip()
    if repo_commits[name] != commit:
        raise RuntimeError(f"{name} commit mismatch")
print(repo_commits)

rawboost_path = REPO_ROOT / "rawboost" / "RawBoost.py"
rawboost_spec = importlib.util.spec_from_file_location("official_rawboost", rawboost_path)
RAWBOOST = importlib.util.module_from_spec(rawboost_spec)
assert rawboost_spec.loader is not None
rawboost_spec.loader.exec_module(RAWBOOST)


def decode_audio(path):
    path = str(path)
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.mean(axis=1))
    except Exception:
        try:
            audio, sample_rate = librosa.load(path, sr=None, mono=True)
            waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32))
        except Exception:
            decoded = subprocess.run(
                [
                    "ffmpeg", "-hide_banner", "-loglevel", "error", "-i", path,
                    "-ac", "1", "-ar", str(CFG.sample_rate), "-f", "f32le", "pipe:1",
                ],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=90,
            )
            waveform = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())
            sample_rate = CFG.sample_rate
    if waveform.numel() == 0:
        raise ValueError(f"empty audio: {path}")
    if sample_rate != CFG.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, CFG.sample_rate)
    waveform = waveform.float().nan_to_num().clamp(-1, 1)
    if waveform.numel() == 0:
        raise ValueError(f"empty after resample: {path}")
    return waveform


def crop_or_repeat(waveform, rng, training):
    if waveform.numel() < CFG.clip_samples:
        waveform = waveform.repeat(math.ceil(CFG.clip_samples / waveform.numel()))
    maximum_start = waveform.numel() - CFG.clip_samples
    start = rng.randint(0, maximum_start) if training and maximum_start else maximum_start // 2
    return waveform[start:start + CFG.clip_samples].clone()


## 8. 출처 편향 진단과 동적 믹싱 전 공통 전처리

원본 파일을 새로 저장하지 않고 각 성분을 불러온 직후, 동적 믹싱 **전에** REAL/FAKE와 무관하게 동일한 확률의 전처리를 적용합니다.

- **코덱/대역:** 8/12/24 kHz round-trip, μ-law, PCM 양자화, band-limit를 클래스 공통 확률로 적용
- **음량:** 모든 성분을 동일한 target RMS 분포로 정규화한 뒤, 두 성분 혼합에서는 별도의 SNR만 부여
- **길이:** 모든 클래스에 동일한 64,600 sample crop/repeat 정책 적용
- **장르/스타일:** FMA `genre_top`과 SONICS `source|algorithm` 그룹을 균형 선택하고, 학습 중에도 그룹을 균등 샘플링
- **진단:** 풀별 원본 확장자·sample rate·duration·RMS 표본을 Drive CSV로 저장

SONICS에는 FMA와 직접 대응되는 공통 장르 라벨이 없으므로 완전한 장르 매칭이라고 과장하지 않습니다. 대신 알려진 장르/생성기 그룹의 편중을 줄이고, 클래스 공통 EQ·대역 변형으로 장르와 인코딩 지름길을 약화합니다. Validation에는 확률적 증강을 적용하지 않고 DC 제거와 고정 길이만 사용합니다.


In [ ]:
BIAS_MITIGATION = SimpleNamespace(
    enabled=True,
    codec_p=0.45,
    eq_p=0.35,
    noise_p=0.08,
    target_db_min=-26.0,
    target_db_max=-18.0,
)
RUN_SOURCE_BIAS_AUDIT = True
SOURCE_BIAS_AUDIT_PER_POOL = 64
SOURCE_BIAS_AUDIT_PATH = MANIFEST_ROOT / "source_bias_audit.csv"


RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"],
    "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"],
    "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"],
    "fv_fm": ["fake_voice", "fake_music"],
}


def rms_normalize(waveform, target_db):
    rms = waveform.square().mean().clamp_min(1e-8).sqrt()
    target = 10 ** (target_db / 20)
    return waveform * (target / rms)


def common_codec_augment(waveform, rng):
    """원본 데이터셋과 무관한 공통 전송/코덱 프로필을 한 번 덧씌운다."""
    mode = rng.choices(
        ["clean", "telephone", "resample", "mulaw", "pcm", "bandlimit"],
        weights=[15, 15, 20, 15, 15, 20],
        k=1,
    )[0]
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "resample":
        intermediate_rate = rng.choice([10_000, 12_000, 22_050, 24_000])
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, intermediate_rate)
        waveform = torchaudio.functional.resample(waveform, intermediate_rate, CFG.sample_rate)
    elif mode == "mulaw":
        channels = rng.choice([128, 256, 512])
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), channels)
        waveform = torchaudio.functional.mu_law_decoding(encoded, channels)
    elif mode == "pcm":
        bits = rng.choice([8, 10, 12, 14])
        levels = float(2 ** (bits - 1) - 1)
        waveform = torch.round(waveform.clamp(-1, 1) * levels) / levels
    elif mode == "bandlimit":
        cutoff = rng.uniform(3400.0, 7600.0)
        waveform = torchaudio.functional.lowpass_biquad(waveform, CFG.sample_rate, cutoff)
        if rng.random() < 0.5:
            waveform = torchaudio.functional.highpass_biquad(
                waveform, CFG.sample_rate, rng.uniform(25.0, 120.0)
            )
    return waveform, mode


def pre_mix_harmonize(waveform, rng, training):
    """길이 통일 후, 믹싱 전에 모든 source class에 같은 분포의 변형을 적용한다."""
    waveform = waveform.float().nan_to_num()
    waveform = waveform - waveform.mean()
    if BIAS_MITIGATION.enabled and training:
        if rng.random() < BIAS_MITIGATION.codec_p:
            waveform, _ = common_codec_augment(waveform, rng)
        if rng.random() < BIAS_MITIGATION.eq_p:
            center = rng.choice([125.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 6500.0])
            waveform = torchaudio.functional.equalizer_biquad(
                waveform, CFG.sample_rate, center, rng.uniform(-6.0, 6.0), rng.uniform(0.5, 1.5)
            )
        if rng.random() < BIAS_MITIGATION.noise_p:
            power = waveform.square().mean().clamp_min(1e-8)
            snr_db = rng.uniform(22.0, 42.0)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
            waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
        target_db = rng.uniform(BIAS_MITIGATION.target_db_min, BIAS_MITIGATION.target_db_max)
    else:
        target_db = -22.0
    waveform = rms_normalize(waveform, target_db)
    return waveform.nan_to_num().clamp(-1, 1)


def inspect_source_bias_row(record):
    path = str(record["path"])
    try:
        info = sf.info(path)
        original_rate = int(info.samplerate)
        channels = int(info.channels)
        codec = f"{info.format}|{info.subtype}"
    except Exception:
        original_rate = -1
        channels = -1
        codec = Path(path).suffix.lower() or "unknown"
    waveform = decode_audio(path)
    duration_seconds = waveform.numel() / CFG.sample_rate
    segment = crop_or_repeat(
        waveform, random.Random(stable_int(f"bias-audit|{record['source_id']}")), training=False
    )
    rms_db = float(20 * torch.log10(segment.square().mean().clamp_min(1e-12).sqrt()))
    return {
        "source_id": record["source_id"], "pool": record["pool"],
        "codec": codec, "original_sample_rate": original_rate, "channels": channels,
        "duration_seconds": duration_seconds, "center_rms_db": rms_db,
        "balance_group": record.get("balance_group", "unknown"),
    }


if RUN_SOURCE_BIAS_AUDIT:
    audit_rows = []
    for pool_name, pool_frame in source_manifest.groupby("pool", sort=False):
        records = pool_frame.to_dict("records")
        records = sorted(
            records, key=lambda row: stable_int(f"bias-audit-select|{CFG.seed}|{row['source_id']}")
        )[:SOURCE_BIAS_AUDIT_PER_POOL]
        for record in tqdm(records, desc=f"source bias audit: {pool_name}"):
            try:
                audit_rows.append(inspect_source_bias_row(record))
            except Exception as exc:
                print("bias audit skip:", record["path"], repr(exc))
    source_bias_audit = pd.DataFrame(audit_rows)
    if not source_bias_audit.empty:
        source_bias_audit.to_csv(SOURCE_BIAS_AUDIT_PATH, index=False, encoding="utf-8")
        display(source_bias_audit.groupby("pool")[[
            "original_sample_rate", "channels", "duration_seconds", "center_rms_db"
        ]].agg(["count", "mean", "std", "median"]).round(3))
        display(pd.crosstab(source_bias_audit["pool"], source_bias_audit["codec"], margins=True))
        print("saved:", SOURCE_BIAS_AUDIT_PATH)


def apply_temporal_layout(waveforms, rng):
    if len(waveforms) < 2:
        return waveforms, "single"
    choice = rng.random()
    if choice < 0.65:
        return waveforms, "overlap"
    length = CFG.clip_samples
    if choice < 0.85:
        result = []
        for waveform in waveforms:
            active = rng.randint(CFG.sample_rate, length)
            start = rng.randint(0, length - active)
            mask = torch.zeros(length)
            mask[start:start + active] = 1
            result.append(waveform * mask)
        return result, "partial"
    boundary = rng.randint(int(0.35 * length), int(0.65 * length))
    first_mask = torch.zeros(length); first_mask[:boundary] = 1
    second_mask = 1 - first_mask
    return [waveforms[0] * first_mask, waveforms[1] * second_mask], "sequential"


def communication_augment(waveform, rng):
    mode = rng.choice(["telephone", "mulaw", "noise", "gain_clip"])
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "mulaw":
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
    elif mode == "noise":
        power = waveform.square().mean().clamp_min(1e-8)
        snr_db = rng.uniform(12, 35)
        generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
        noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
        waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
    else:
        waveform = waveform * 10 ** (rng.uniform(-8, 5) / 20)
        limit = rng.uniform(0.35, 0.95)
        waveform = waveform.clamp(-limit, limit) / limit
    return waveform


class DynamicMixDataset(Dataset):
    def __init__(self, recipes, sources, training=False, rawboost_p=0.0, communication_p=0.0):
        self.recipes = recipes.reset_index(drop=True)
        self.training = training
        self.rawboost_p = rawboost_p
        self.communication_p = communication_p
        self.epoch = 0
        self.pools = {
            pool: frame.to_dict("records")
            for pool, frame in sources.groupby("pool", sort=False)
        }
        self.pool_groups = {}
        for pool, records in self.pools.items():
            groups = {}
            for record in records:
                group = str(record.get("balance_group", "unknown") or "unknown")
                groups.setdefault(group, []).append(record)
            self.pool_groups[pool] = [groups[name] for name in sorted(groups)]
        for pool in RECIPE_COMPONENTS.values():
            for name in pool:
                if not self.pools.get(name):
                    raise RuntimeError(f"empty source pool: {name}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.recipes)

    def _load_from_pool(self, pool_name, rng):
        errors = []
        for _ in range(5):
            groups = self.pool_groups[pool_name]
            group_records = groups[rng.randrange(len(groups))]
            entry = group_records[rng.randrange(len(group_records))]
            try:
                waveform = crop_or_repeat(decode_audio(entry["path"]), rng, self.training)
                waveform = pre_mix_harmonize(waveform, rng, self.training)
                return waveform, entry
            except Exception as exc:
                errors.append(f"{entry['path']}: {repr(exc)}")
        raise RuntimeError(" | ".join(errors))

    def __getitem__(self, index):
        recipe = self.recipes.iloc[index]
        epoch = self.epoch if self.training else 0
        rng = random.Random(stable_int(f"mix|{recipe.seed}|{epoch}|{index}"))
        components = RECIPE_COMPONENTS[recipe.recipe_type]
        waveforms, entries = [], []
        try:
            for pool_name in components:
                waveform, entry = self._load_from_pool(pool_name, rng)
                waveforms.append(waveform)
                entries.append((pool_name, entry))
        except Exception as exc:
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": torch.zeros(5),
                "mask": torch.ones(5), "id": recipe.recipe_id,
                "recipe_type": recipe.recipe_type, "valid": torch.tensor(False),
                "error": repr(exc), "layout": "error",
            }

        if len(waveforms) == 2:
            snr_db = rng.uniform(-12, 12)
            waveforms[0] = rms_normalize(waveforms[0], -22 + snr_db / 2)
            waveforms[1] = rms_normalize(waveforms[1], -22 - snr_db / 2)
        else:
            waveforms[0] = rms_normalize(waveforms[0], rng.uniform(-27, -17))
        waveforms, layout = apply_temporal_layout(waveforms, rng)
        mixed = torch.stack(waveforms).sum(0)

        target = torch.zeros(5, dtype=torch.float32)
        voice_present = voice_fake = music_present = music_fake = 0
        for pool_name, entry in entries:
            if pool_name.endswith("voice"):
                voice_present = 1
                voice_fake = max(voice_fake, int(pool_name == "fake_voice"))
            if pool_name.endswith("music"):
                music_present = 1
                music_fake = max(music_fake, int(pool_name == "fake_music"))
                if bool(entry.get("contains_voice", False)):
                    voice_present = 1
                    voice_fake = max(voice_fake, int(pool_name == "fake_music"))
        file_fake = max(voice_fake, music_fake)
        target[:] = torch.tensor([file_fake, voice_fake, music_fake, voice_present, music_present])
        mask = torch.tensor([1, voice_present, music_present, 1, 1], dtype=torch.float32)

        if self.training and rng.random() < self.rawboost_p:
            values = mixed.numpy()
            values = RAWBOOST.LnL_convolutive_noise(
                values, N_f=5, nBands=5, minF=20, maxF=8000,
                minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
                minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20,
                fs=CFG.sample_rate,
            )
            values = RAWBOOST.ISD_additive_noise(values, P=10, g_sd=2)
            mixed = torch.from_numpy(np.asarray(RAWBOOST.normWav(values, 0), dtype=np.float32))
        if self.training and rng.random() < self.communication_p:
            mixed = communication_augment(mixed, rng)

        mixed = mixed - mixed.mean()
        peak = mixed.abs().max().clamp_min(1e-8)
        mixed = mixed / peak * rng.uniform(0.65, 0.98)
        if not torch.isfinite(mixed).all():
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": target, "mask": mask,
                "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
                "valid": torch.tensor(False), "error": "NaN/Inf after mixing", "layout": layout,
            }
        return {
            "audio": mixed.float().clamp(-1, 1), "target": target, "mask": mask,
            "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
            "valid": torch.tensor(True), "error": "", "layout": layout,
        }


source_by_split = {
    split_name: source_manifest[source_manifest["split"] == split_name].copy()
    for split_name in CFG.source_split_counts
}
recipe_by_split = {
    split_name: recipe_manifest[recipe_manifest["split"] == split_name].copy()
    for split_name in CFG.recipe_counts
}
preview_dataset = DynamicMixDataset(recipe_by_split["validation"].head(8), source_by_split["validation"])
preview_rows = []
for index in range(len(preview_dataset)):
    item = preview_dataset[index]
    preview_rows.append({"id": item["id"], "type": item["recipe_type"], **dict(zip(DACON_TRUTH_COLUMNS, item["target"].tolist()))})
display(pd.DataFrame(preview_rows))


## 9. DACON 공식 지표와 masked multi-task loss

[DACON 공식 평가 페이지](https://dacon.io/competitions/official/236749/overview/evaluation)의 계산을 그대로 사용합니다.

- `ADS = 0.5 × (1 - File EER) + 0.2 × (1 - Voice EER) + 0.3 × (1 - Music EER)`
- `CPS = 0.5 × Voice Presence ROC-AUC + 0.5 × Music Presence ROC-AUC`
- `Score = 0.9 × ADS + 0.1 × CPS` (높을수록 좋음)
- FAKE가 양성 클래스 `1`이며, Voice/Music EER은 해당 성분이 존재하는 샘플에서만 계산합니다.

Loss weight는 최종 Score의 각 항 가중치를 그대로 펼친 값입니다: File 0.45, Voice Fake 0.18, Music Fake 0.27, Voice Presence 0.05, Music Presence 0.05. 성분이 없는 fake head는 loss mask에서 제외합니다.


In [ ]:
OFFICIAL_METRIC_WEIGHTS = {
    "score_ads": 0.9,
    "score_cps": 0.1,
    "ads_file": 0.5,
    "ads_voice": 0.2,
    "ads_music": 0.3,
    "cps_voice_presence": 0.5,
    "cps_music_presence": 0.5,
}


def _official_binary_inputs(y_true, y_score, metric_name):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.shape != y_score.shape or y_true.size == 0:
        raise ValueError(f"{metric_name}: shape/empty input error")
    if not np.isfinite(y_true).all() or not np.isfinite(y_score).all():
        raise ValueError(f"{metric_name}: NaN/Inf input")
    if np.unique(y_true).size < 2:
        raise ValueError(f"{metric_name}: positive/negative classes are both required")
    return y_true, y_score


def equal_error_rate(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "EER")
    # DACON 평가 페이지에 공개된 EER 구현과 동일합니다.
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return float(eer)


def official_roc_auc(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "ROC-AUC")
    return float(roc_auc_score(y_true, y_score))


def dacon_official_score(y_true, y_pred):
    file_eer = equal_error_rate(y_true["FILE_FAKE"], y_pred["FILE_FAKE_PROB"])
    voice_mask = y_true["VOICE_PRESENT"].eq(1)
    music_mask = y_true["MUSIC_PRESENT"].eq(1)
    voice_eer = equal_error_rate(y_true.loc[voice_mask, "VOICE_FAKE"], y_pred.loc[voice_mask, "VOICE_FAKE_PROB"])
    music_eer = equal_error_rate(y_true.loc[music_mask, "MUSIC_FAKE"], y_pred.loc[music_mask, "MUSIC_FAKE_PROB"])
    voice_auc = official_roc_auc(y_true["VOICE_PRESENT"], y_pred["VOICE_PRESENT_PROB"])
    music_auc = official_roc_auc(y_true["MUSIC_PRESENT"], y_pred["MUSIC_PRESENT_PROB"])
    ads = (
        OFFICIAL_METRIC_WEIGHTS["ads_file"] * (1 - file_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_voice"] * (1 - voice_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_music"] * (1 - music_eer)
    )
    cps = (
        OFFICIAL_METRIC_WEIGHTS["cps_voice_presence"] * voice_auc
        + OFFICIAL_METRIC_WEIGHTS["cps_music_presence"] * music_auc
    )
    score = (
        OFFICIAL_METRIC_WEIGHTS["score_ads"] * ads
        + OFFICIAL_METRIC_WEIGHTS["score_cps"] * cps
    )
    return {
        "file_eer": file_eer, "voice_eer": voice_eer, "music_eer": music_eer,
        "voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
        "ads": ads, "cps": cps, "score": score,
    }


def masked_multitask_loss(logits, targets, masks):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    weights = HEAD_WEIGHTS.to(logits.device).unsqueeze(0) * masks
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)


def report_from_arrays(targets, predictions):
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    y_pred = pd.DataFrame(predictions, columns=DACON_PROBABILITY_COLUMNS)
    return dacon_official_score(y_true, y_pred)


## 9.1 클래스 공통 MP3/AAC/Opus·reverb·clipping 보강

REAL/FAKE 어느 쪽에도 같은 확률을 사용합니다. 실제 FFmpeg 메모리 round-trip은 MP3/AAC/Opus와 bitrate를 무작위로 고르며, 실패하면 원본을 유지해 DataLoader가 중단되지 않습니다. clean sample도 충분히 남도록 전체 확률은 제한합니다.


In [ ]:
BIAS_MITIGATION.reverb_p = 0.08
BIAS_MITIGATION.clipping_p = 0.08
_FFMPEG_CODEC_FAILURES = 0


def ffmpeg_codec_roundtrip(waveform, rng):
    global _FFMPEG_CODEC_FAILURES
    choices = [
        ("mp3", "libmp3lame", bitrate) for bitrate in ("32k", "48k", "64k", "96k")
    ] + [
        ("adts", "aac", bitrate) for bitrate in ("32k", "48k", "64k", "96k")
    ] + [
        ("ogg", "libopus", bitrate) for bitrate in ("24k", "32k", "48k", "64k")
    ]
    container, encoder, bitrate = rng.choice(choices)
    raw = np.asarray(waveform.detach().cpu(), dtype="<f4").tobytes()
    try:
        encoded = subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-f", "f32le", "-ar", "16000",
             "-ac", "1", "-i", "pipe:0", "-c:a", encoder, "-b:a", bitrate, "-f", container, "pipe:1"],
            input=raw, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=30,
        ).stdout
        decoded = subprocess.run(
            ["ffmpeg", "-hide_banner", "-loglevel", "error", "-i", "pipe:0", "-f", "f32le",
             "-ar", "16000", "-ac", "1", "pipe:1"],
            input=encoded, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=30,
        ).stdout
        result = torch.from_numpy(np.frombuffer(decoded, dtype="<f4").copy())
        if result.numel() < waveform.numel():
            result = result.repeat(math.ceil(waveform.numel() / max(1, result.numel())))
        result = result[:waveform.numel()]
        if result.numel() != waveform.numel() or not torch.isfinite(result).all():
            raise ValueError("invalid codec round-trip")
        return result, f"{encoder}:{bitrate}"
    except Exception as exc:
        _FFMPEG_CODEC_FAILURES += 1
        if _FFMPEG_CODEC_FAILURES <= 3:
            print("codec round-trip fallback:", repr(exc))
        return waveform, "lossy_fallback"


def common_codec_augment(waveform, rng):
    mode = rng.choices(
        ["clean", "telephone", "resample", "mulaw", "pcm", "bandlimit", "lossy"],
        weights=[25, 15, 15, 10, 10, 10, 15], k=1,
    )[0]
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "resample":
        intermediate_rate = rng.choice([10_000, 12_000, 22_050, 24_000])
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, intermediate_rate)
        waveform = torchaudio.functional.resample(waveform, intermediate_rate, CFG.sample_rate)
    elif mode == "mulaw":
        channels = rng.choice([128, 256, 512])
        waveform = torchaudio.functional.mu_law_decoding(
            torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), channels), channels
        )
    elif mode == "pcm":
        levels = float(2 ** (rng.choice([8, 10, 12, 14]) - 1) - 1)
        waveform = torch.round(waveform.clamp(-1, 1) * levels) / levels
    elif mode == "bandlimit":
        waveform = torchaudio.functional.lowpass_biquad(
            waveform, CFG.sample_rate, rng.uniform(3400.0, 7600.0)
        )
        if rng.random() < .5:
            waveform = torchaudio.functional.highpass_biquad(
                waveform, CFG.sample_rate, rng.uniform(25.0, 120.0)
            )
    elif mode == "lossy":
        waveform, mode = ffmpeg_codec_roundtrip(waveform, rng)
    return waveform, mode


def light_reverb(waveform, rng):
    result = waveform.clone()
    for delay_ms in rng.sample([18, 31, 47, 71, 103, 137], k=3):
        delay = int(delay_ms * CFG.sample_rate / 1000)
        gain = rng.uniform(.06, .22) * rng.choice([-1, 1])
        result[delay:] += gain * waveform[:-delay]
    return result / result.abs().max().clamp_min(1.0)


def pre_mix_harmonize(waveform, rng, training):
    waveform = waveform.float().nan_to_num() - waveform.float().nan_to_num().mean()
    if BIAS_MITIGATION.enabled and training:
        if rng.random() < BIAS_MITIGATION.codec_p:
            waveform, _ = common_codec_augment(waveform, rng)
        if rng.random() < BIAS_MITIGATION.eq_p:
            waveform = torchaudio.functional.equalizer_biquad(
                waveform, CFG.sample_rate,
                rng.choice([125.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 6500.0]),
                rng.uniform(-6.0, 6.0), rng.uniform(0.5, 1.5),
            )
        if rng.random() < BIAS_MITIGATION.noise_p:
            power = waveform.square().mean().clamp_min(1e-8)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
            waveform += noise * (power / 10 ** (rng.uniform(22.0, 42.0) / 10)).sqrt()
        if rng.random() < BIAS_MITIGATION.reverb_p:
            waveform = light_reverb(waveform, rng)
        if rng.random() < BIAS_MITIGATION.clipping_p:
            limit = rng.uniform(.45, .95)
            waveform = waveform.clamp(-limit, limit) / limit
        target_db = rng.uniform(BIAS_MITIGATION.target_db_min, BIAS_MITIGATION.target_db_max)
    else:
        target_db = -22.0
    return rms_normalize(waveform, target_db).nan_to_num().clamp(-1, 1)


## 10. Domain-shift stress validation view

동일한 held-out source와 recipe를 사용하되 stress view에는 학습 epoch와 무관한 고정 seed로 telephone, resampling, μ-law/PCM, band-limit, EQ, noise, clipping 중 여러 변형을 적용합니다. 따라서 source 수와 recipe 수는 늘리지 않으면서 clean 2,500과 stress 2,500 성능을 함께 확인합니다.


In [ ]:
class DomainStressView(Dataset):
    def __init__(self, base_dataset, enabled):
        self.base = base_dataset
        self.enabled = bool(enabled)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        item = self.base[index]
        if not self.enabled or not bool(item["valid"]):
            return item
        rng = random.Random(stable_int(f"domain-stress|{CFG.seed}|{item['id']}"))
        audio = item["audio"].clone()
        # 학습 augmentation과 다른 조합 강도. 결과는 index별로 항상 동일하다.
        audio, codec_mode = common_codec_augment(audio, rng)
        if rng.random() < 0.70:
            center = rng.choice([180.0, 350.0, 750.0, 1500.0, 3000.0, 6000.0])
            audio = torchaudio.functional.equalizer_biquad(
                audio, CFG.sample_rate, center, rng.uniform(-8.0, 8.0), rng.uniform(0.4, 1.8)
            )
        if rng.random() < 0.45:
            audio = communication_augment(audio, rng)
        if rng.random() < 0.35:
            power = audio.square().mean().clamp_min(1e-8)
            snr_db = rng.uniform(10.0, 28.0)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(audio.shape, generator=generator, dtype=audio.dtype)
            audio = audio + noise * (power / 10 ** (snr_db / 10)).sqrt()
        audio = rms_normalize(audio - audio.mean(), rng.uniform(-27.0, -17.0))
        limit = rng.uniform(0.55, 0.98)
        audio = audio.clamp(-limit, limit) / limit
        result = dict(item)
        result["audio"] = audio.nan_to_num().float().clamp(-1, 1)
        result["layout"] = f"stress:{codec_mode}"
        return result


validation_base = DynamicMixDataset(
    recipe_by_split["validation"], source_by_split["validation"], training=False
)
validation_clean = DomainStressView(validation_base, enabled=False)
validation_stress = DomainStressView(validation_base, enabled=True)
print("validation views:", len(validation_clean), len(validation_stress))


## 11. 역할이 분리된 모델

- `PresenceLogMel`: Voice/Music 존재 확률 2개만 학습
- `AASISTFake3`: File/Voice/Music Fake 확률 3개만 학습
- `XLSRDualGraphFake3`: 같은 Fake 3개를 다른 representation으로 학습

두 Fake 모델이 서로 다른 오류를 내도록 backbone과 head 구조를 분리합니다.


In [ ]:
class PresenceLogMel(nn.Module):
    def __init__(self, dropout=0.20):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, win_length=400,
            hop_length=160, n_mels=96, f_min=20, f_max=7600,
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 192, 3, padding=1), nn.BatchNorm2d(192), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(192, 2))

    def forward(self, audio):
        feature = torch.log(self.mel(audio).clamp_min(1e-6))
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (
            feature.std((-2, -1), keepdim=True) + 1e-5
        )
        return self.head(self.encoder(feature.unsqueeze(1)))


def build_aasist_fake3():
    repo = REPO_ROOT / "aasist"
    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as OfficialAASIST

    class AASISTFake3(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = OfficialAASIST(model_config)
            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 3)

        def forward(self, audio):
            _, logits = self.net(audio, Freq_aug=False)
            return logits

    return AASISTFake3()


from transformers import AutoModel
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"
XLSR_REVISION = "1a640f32ac3e39899438a2931f9924c02f080a54"


class SSLBase(nn.Module):
    def __init__(self, model_name, freeze=True):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name, revision=XLSR_REVISION)
        self.hidden = self.ssl.config.hidden_size
        if freeze:
            for parameter in self.ssl.parameters():
                parameter.requires_grad = False

    def features(self, audio):
        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)
        if any(parameter.requires_grad for parameter in self.ssl.parameters()):
            return self.ssl(audio).last_hidden_state
        with torch.no_grad():
            return self.ssl(audio).last_hidden_state

    def unfreeze_last(self, count=4):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False
        for layer in self.ssl.encoder.layers[-count:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True


class AttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, 4, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attention(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRDualGraphFake3(SSLBase):
    def __init__(self, model_name=XLSR_MODEL, freeze=True, dim=128, dropout=0.20):
        super().__init__(model_name, freeze)
        self.projection = nn.Linear(self.hidden, dim)
        self.feature_projection = nn.Linear(self.hidden, dim)
        self.time_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.feature_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.head = nn.Sequential(
            nn.LayerNorm(dim * 4), nn.Dropout(dropout), nn.Linear(dim * 4, 256),
            nn.GELU(), nn.Linear(256, 3),
        )

    def forward(self, audio):
        raw = self.features(audio)
        hidden = self.projection(raw)
        time_nodes = F.adaptive_avg_pool1d(hidden.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = self.feature_projection(
            F.adaptive_max_pool1d(raw.transpose(1, 2), 8).transpose(1, 2)
        )
        time_nodes = self.time_graph(time_nodes)
        feature_nodes = self.feature_graph(feature_nodes)
        pooled = torch.cat([
            time_nodes.mean(1), time_nodes.amax(1),
            feature_nodes.mean(1), feature_nodes.amax(1),
        ], dim=-1)
        return self.head(pooled)


MODEL_CONFIGS = {
    "presence": dict(task="presence", batch=32, eval_batch=64, grad_accum=1, epochs=10,
                     lr=3e-4, backbone_lr=3e-4, patience=4, rawboost_p=0.03,
                     communication_p=0.08, dropout=0.20),
    "aasist_fake3": dict(task="fake", batch=16, eval_batch=32, grad_accum=1, epochs=16,
                         lr=1e-4, backbone_lr=1e-4, patience=5, rawboost_p=0.20,
                         communication_p=0.10, dropout=0.20, ranking_weight=0.08),
    "xlsr_dualgraph_fake3": dict(task="fake", batch=2, eval_batch=4, grad_accum=8, epochs=10,
                                 lr=8e-5, backbone_lr=3e-7, patience=4, freeze_epochs=3,
                                 unfreeze_last=4, rawboost_p=0.20, communication_p=0.10,
                                 dropout=0.20, ranking_weight=0.08),
}
display(pd.DataFrame(MODEL_CONFIGS).T)


## 12. Masked BCE + ranking loss와 clean/stress 동시 평가

Voice Fake loss는 Voice가 있는 sample에서만, Music Fake loss는 Music이 있는 sample에서만 계산합니다. Ranking loss는 같은 batch 안에서 fake의 logit이 real보다 높도록 보조합니다.


In [ ]:
def pairwise_ranking_loss(logits, targets, masks, margin=0.25):
    losses = []
    for head in range(logits.shape[1]):
        valid = masks[:, head].bool()
        positive = logits[valid & targets[:, head].eq(1), head]
        negative = logits[valid & targets[:, head].eq(0), head]
        if positive.numel() and negative.numel():
            losses.append(F.softplus(margin - (positive[:, None] - negative[None, :])).mean())
    return torch.stack(losses).mean() if losses else logits.sum() * 0


def task_targets(batch, task):
    if task == "presence":
        return batch["target"][:, 3:5], torch.ones_like(batch["target"][:, 3:5])
    return batch["target"][:, :3], batch["mask"][:, :3]


def task_loss(logits, targets, masks, task, ranking_weight=0.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    bce = (bce * masks).sum() / masks.sum().clamp_min(1)
    if task == "fake" and ranking_weight:
        bce = bce + ranking_weight * pairwise_ranking_loss(logits, targets, masks)
    return bce


def fake_report(targets, predictions, masks):
    eers = []
    for head, name in enumerate(("file_eer", "voice_eer", "music_eer")):
        valid = masks[:, head].astype(bool)
        eers.append((name, equal_error_rate(targets[valid, head], predictions[valid, head])))
    report = dict(eers)
    report["ads"] = .5 * (1 - report["file_eer"]) + .2 * (1 - report["voice_eer"]) + .3 * (1 - report["music_eer"])
    return report


def presence_report(targets, predictions):
    voice_auc = official_roc_auc(targets[:, 0], predictions[:, 0])
    music_auc = official_roc_auc(targets[:, 1], predictions[:, 1])
    return {"voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
            "cps": .5 * voice_auc + .5 * music_auc}


def build_branch(name):
    if name == "presence":
        return PresenceLogMel(MODEL_CONFIGS[name]["dropout"])
    if name == "aasist_fake3":
        return build_aasist_fake3()
    if name == "xlsr_dualgraph_fake3":
        return XLSRDualGraphFake3(dropout=MODEL_CONFIGS[name]["dropout"])
    raise KeyError(name)


def make_branch_loaders(config):
    train_dataset = DynamicMixDataset(
        recipe_by_split["train"], source_by_split["train"], training=True,
        rawboost_p=config["rawboost_p"], communication_p=config["communication_p"],
    )
    common = dict(num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda", persistent_workers=False)
    return (
        train_dataset,
        DataLoader(train_dataset, batch_size=config["batch"], shuffle=True, drop_last=True, **common),
        DataLoader(validation_clean, batch_size=config["eval_batch"], shuffle=False, **common),
        DataLoader(validation_stress, batch_size=config["eval_batch"], shuffle=False, **common),
    )


def run_branch_loader(model, loader, config, optimizer=None, scheduler=None, scaler=None, description="eval"):
    training = optimizer is not None
    model.train(training)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = count = skipped = 0
    all_targets, all_predictions, all_masks, all_ids = [], [], [], []
    for step, batch in enumerate(tqdm(loader, desc=description, dynamic_ncols=True), 1):
        valid = batch["valid"].bool()
        skipped += int((~valid).sum())
        if not valid.any():
            continue
        audio = batch["audio"][valid].to(DEVICE, non_blocking=True)
        raw_targets, raw_masks = task_targets(batch, config["task"])
        targets = raw_targets[valid].to(DEVICE, non_blocking=True)
        masks = raw_masks[valid].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(audio)
            unscaled = task_loss(logits, targets, masks, config["task"], config.get("ranking_weight", 0.0))
            loss = unscaled / (config["grad_accum"] if training else 1)
        if training:
            scaler.scale(loss).backward()
            if step % config["grad_accum"] == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        batch_count = int(targets.shape[0])
        total_loss += float(unscaled.detach().cpu()) * batch_count
        count += batch_count
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(torch.sigmoid(logits.detach()).cpu().numpy())
        all_masks.append(masks.detach().cpu().numpy())
        flags = valid.tolist()
        all_ids.extend([value for value, keep in zip(batch["id"], flags) if keep])
    targets = np.concatenate(all_targets)
    predictions = np.concatenate(all_predictions)
    masks = np.concatenate(all_masks)
    report = presence_report(targets, predictions) if config["task"] == "presence" else fake_report(targets, predictions, masks)
    report.update(loss=total_loss / max(count, 1), samples=count, skipped=skipped)
    return report, targets, predictions, masks, all_ids


## 13. 세 branch 학습·resume·best checkpoint

모델 선택 기준은 clean과 stress validation 점수의 평균입니다. Presence는 CPS, Fake는 ADS를 사용합니다.


In [ ]:
RESUME_TRAINING = True


def fit_branch(name):
    from transformers import get_cosine_schedule_with_warmup
    config = dict(MODEL_CONFIGS[name])
    run_dir = RUN_ROOT / name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_dataset, train_loader, clean_loader, stress_loader = make_branch_loaders(config)
    model = build_branch(name).to(DEVICE)
    backbone, head = [], []
    for parameter_name, parameter in model.named_parameters():
        (backbone if parameter_name.startswith("ssl.") else head).append(parameter)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": config["backbone_lr"]})
    if head:
        groups.append({"params": head, "lr": config["lr"]})
    optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
    updates = math.ceil(len(train_loader) / config["grad_accum"]) * config["epochs"]
    scheduler = get_cosine_schedule_with_warmup(optimizer, max(10, int(updates * .08)), updates)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

    best_score, stale, history, start_epoch = -float("inf"), 0, [], 1
    last_path = run_dir / "last.pt"
    if RESUME_TRAINING and last_path.exists():
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        if state.get("config") == config:
            model.load_state_dict(state["model_state"], strict=True)
            optimizer.load_state_dict(state["optimizer_state"])
            scheduler.load_state_dict(state["scheduler_state"])
            scaler.load_state_dict(state["scaler_state"])
            best_score, stale = float(state["best_score"]), int(state["stale"])
            history, start_epoch = list(state["history"]), int(state["epoch"]) + 1
            if state.get("finished"):
                return pd.DataFrame(history)

    if hasattr(model, "unfreeze_last") and start_epoch > config.get("freeze_epochs", float("inf")):
        model.unfreeze_last(config.get("unfreeze_last", 4))

    for epoch in range(start_epoch, config["epochs"] + 1):
        started = time.time()
        train_dataset.set_epoch(epoch)
        if epoch == config.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last"):
            model.unfreeze_last(config.get("unfreeze_last", 4))
        train_report, *_ = run_branch_loader(
            model, train_loader, config, optimizer, scheduler, scaler, f"{name} train {epoch}"
        )
        with torch.no_grad():
            clean = run_branch_loader(model, clean_loader, config, description=f"{name} clean val")
            stress = run_branch_loader(model, stress_loader, config, description=f"{name} stress val")
        metric_key = "cps" if config["task"] == "presence" else "ads"
        robust_score = .5 * clean[0][metric_key] + .5 * stress[0][metric_key]
        record = {
            "epoch": epoch, "train_loss": train_report["loss"], "robust_score": robust_score,
            **{f"clean_{k}": v for k, v in clean[0].items()},
            **{f"stress_{k}": v for k, v in stress[0].items()},
            "minutes": (time.time() - started) / 60,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print(record)
        if robust_score > best_score:
            best_score, stale = robust_score, 0
            torch.save({
                "model_state": model.state_dict(), "name": name, "config": config,
                "epoch": epoch, "robust_score": robust_score,
                "repo_commits": repo_commits,
                "source_manifest_sha256": hashlib.sha256(SOURCE_MANIFEST_PATH.read_bytes()).hexdigest(),
                "recipe_manifest_sha256": hashlib.sha256(RECIPE_MANIFEST_PATH.read_bytes()).hexdigest(),
            }, run_dir / "best.pt")
            for view_name, result in (("clean", clean), ("stress", stress)):
                report, targets, predictions, masks, identifiers = result
                frame = pd.DataFrame({"ID": identifiers})
                prefix = ["VOICE_PRESENT", "MUSIC_PRESENT"] if config["task"] == "presence" else ["FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE"]
                for idx, column in enumerate(prefix):
                    frame[column] = targets[:, idx]
                    frame[column + "_PROB"] = predictions[:, idx]
                    frame[column + "_MASK"] = masks[:, idx]
                frame.to_csv(run_dir / f"validation_{view_name}_predictions.csv", index=False)
        else:
            stale += 1
        finished = stale >= config["patience"] or epoch == config["epochs"]
        torch.save({
            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
            "config": config, "epoch": epoch, "best_score": best_score,
            "stale": stale, "history": history, "finished": finished,
        }, last_path)
        if stale >= config["patience"]:
            break
    del model
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(history)


RUN_TRAINING = True
BRANCHES_TO_TRAIN = ["presence", "aasist_fake3", "xlsr_dualgraph_fake3"]

if RUN_TRAINING:
    summaries = []
    for branch_name in BRANCHES_TO_TRAIN:
        history = fit_branch(branch_name)
        best = history.loc[history.robust_score.idxmax()].to_dict()
        summaries.append({"branch": branch_name, **best})
    display(pd.DataFrame(summaries).sort_values("robust_score", ascending=False))


## 14. Validation-only ensemble weight 선택과 공식 점수

OOD test와 DACON test는 weight 선택에 사용하지 않습니다. AASIST weight를 0~1 사이에서 head별로 탐색하고 clean/stress ADS 평균이 가장 높은 값을 고정합니다.


In [ ]:
FUSION_PATH = RUN_ROOT / "fusion.json"


def load_validation(branch, view):
    return pd.read_csv(RUN_ROOT / branch / f"validation_{view}_predictions.csv")


def blend_probabilities(first, second, weight_first):
    epsilon = 1e-5
    first = np.clip(np.asarray(first, float), epsilon, 1 - epsilon)
    second = np.clip(np.asarray(second, float), epsilon, 1 - epsilon)
    first_logit = np.log(first / (1 - first))
    second_logit = np.log(second / (1 - second))
    value = weight_first * first_logit + (1 - weight_first) * second_logit
    return 1 / (1 + np.exp(-value))


aasist_frames = {view: load_validation("aasist_fake3", view) for view in ("clean", "stress")}
xlsr_frames = {view: load_validation("xlsr_dualgraph_fake3", view) for view in ("clean", "stress")}
presence_frames = {view: load_validation("presence", view) for view in ("clean", "stress")}

fusion_weights = {}
for head in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE"):
    candidates = []
    for weight in np.linspace(0, 1, 21):
        scores = []
        for view in ("clean", "stress"):
            left = aasist_frames[view]
            right = xlsr_frames[view]
            if left.ID.astype(str).tolist() != right.ID.astype(str).tolist():
                raise RuntimeError(f"{view} ID mismatch")
            valid = left[head + "_MASK"].eq(1)
            prediction = blend_probabilities(
                left.loc[valid, head + "_PROB"], right.loc[valid, head + "_PROB"], weight
            )
            scores.append(1 - equal_error_rate(left.loc[valid, head], prediction))
        candidates.append((float(np.mean(scores)), float(weight)))
    fusion_weights[head.lower() + "_aasist_weight"] = max(candidates)[1]

fusion = {
    "weights": fusion_weights,
    "aggregation": {"top_fraction": 0.30, "minimum_segments": 1, "maximum_segments": 8},
    "file_direct_weight": 0.60,
    "selection_data": "held-out validation clean+stress only",
}
FUSION_PATH.write_text(json.dumps(fusion, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(fusion, ensure_ascii=False, indent=2))


def validation_official_report(view):
    presence = presence_frames[view]
    aasist = aasist_frames[view]
    xlsr = xlsr_frames[view]
    truth = pd.DataFrame({
        "FILE_FAKE": aasist.FILE_FAKE,
        "VOICE_FAKE": aasist.VOICE_FAKE,
        "MUSIC_FAKE": aasist.MUSIC_FAKE,
        "VOICE_PRESENT": presence.VOICE_PRESENT,
        "MUSIC_PRESENT": presence.MUSIC_PRESENT,
    })
    prediction = pd.DataFrame({
        "FILE_FAKE_PROB": blend_probabilities(aasist.FILE_FAKE_PROB, xlsr.FILE_FAKE_PROB, fusion_weights["file_fake_aasist_weight"]),
        "VOICE_FAKE_PROB": blend_probabilities(aasist.VOICE_FAKE_PROB, xlsr.VOICE_FAKE_PROB, fusion_weights["voice_fake_aasist_weight"]),
        "MUSIC_FAKE_PROB": blend_probabilities(aasist.MUSIC_FAKE_PROB, xlsr.MUSIC_FAKE_PROB, fusion_weights["music_fake_aasist_weight"]),
        "VOICE_PRESENT_PROB": presence.VOICE_PRESENT_PROB,
        "MUSIC_PRESENT_PROB": presence.MUSIC_PRESENT_PROB,
    })
    return dacon_official_score(truth, prediction)


validation_reports = pd.DataFrame([
    {"view": view, **validation_official_report(view)} for view in ("clean", "stress")
])
display(validation_reports)
validation_reports.to_csv(RUN_ROOT / "validation_ensemble_official_metrics.csv", index=False)


## 15. 긴 파일용 Presence-gated Top-K inference

최대 8개 segment를 시작부터 끝까지 균등 선택합니다. Presence는 상위 30% 평균, 각 component Fake는 해당 Presence가 높은 segment 안에서 다시 상위 score를 평균합니다.


In [ ]:
def load_best_branch(name):
    checkpoint = torch.load(RUN_ROOT / name / "best.pt", map_location="cpu", weights_only=False)
    model = build_branch(name)
    model.load_state_dict(checkpoint["model_state"], strict=True)
    return model.to(DEVICE).eval()


def evaluation_segments(audio, maximum=8):
    starts = list(range(0, max(1, audio.numel() - CFG.clip_samples + 1), CFG.clip_samples))
    last = max(0, audio.numel() - CFG.clip_samples)
    if not starts or starts[-1] != last:
        starts.append(last)
    if len(starts) > maximum:
        indices = np.linspace(0, len(starts) - 1, maximum).round().astype(int)
        starts = [starts[index] for index in sorted(set(indices.tolist()))]
    segments = []
    for start in starts:
        segment = audio[start:start + CFG.clip_samples]
        if segment.numel() < CFG.clip_samples:
            segment = segment.repeat(math.ceil(CFG.clip_samples / max(1, segment.numel())))[:CFG.clip_samples]
        segment = segment - segment.mean()
        peak = segment.abs().max().clamp_min(1e-8)
        segments.append((segment / peak * .82).clamp(-1, 1))
    return torch.stack(segments)


def topk_mean(values, fraction=.30):
    values = np.asarray(values, float)
    count = max(1, int(math.ceil(len(values) * fraction)))
    return float(np.mean(np.sort(values)[-count:]))


def presence_gated_fake(fake_scores, presence_scores, fraction=.30):
    fake_scores = np.asarray(fake_scores, float)
    presence_scores = np.asarray(presence_scores, float)
    gate_count = max(1, int(math.ceil(len(fake_scores) * max(.50, fraction))))
    gated = np.argsort(presence_scores)[-gate_count:]
    return topk_mean(fake_scores[gated], fraction)


def predict_segments(model, segments, batch_size):
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(segments), batch_size):
            batch = segments[start:start + batch_size].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                logits = model(batch)
            outputs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(outputs)


def predict_file_ensemble(audio, models, fusion):
    segments = evaluation_segments(audio, fusion["aggregation"]["maximum_segments"])
    presence = predict_segments(models["presence"], segments, 64)
    aasist = predict_segments(models["aasist_fake3"], segments, 32)
    xlsr = predict_segments(models["xlsr_dualgraph_fake3"], segments, 4)
    fake = np.zeros_like(aasist)
    for head, name in enumerate(("file_fake", "voice_fake", "music_fake")):
        fake[:, head] = blend_probabilities(
            aasist[:, head], xlsr[:, head], fusion["weights"][name + "_aasist_weight"]
        )
    voice_presence = topk_mean(presence[:, 0], fusion["aggregation"]["top_fraction"])
    music_presence = topk_mean(presence[:, 1], fusion["aggregation"]["top_fraction"])
    voice_fake = presence_gated_fake(fake[:, 1], presence[:, 0], fusion["aggregation"]["top_fraction"])
    music_fake = presence_gated_fake(fake[:, 2], presence[:, 1], fusion["aggregation"]["top_fraction"])
    direct_file = topk_mean(fake[:, 0], fusion["aggregation"]["top_fraction"])
    coherent = 1 - (1 - voice_presence * voice_fake) * (1 - music_presence * music_fake)
    file_fake = fusion["file_direct_weight"] * direct_file + (1 - fusion["file_direct_weight"]) * coherent
    return np.clip([file_fake, voice_fake, music_fake, voice_presence, music_presence], 0, 1)


trained_models = {name: load_best_branch(name) for name in BRANCHES_TO_TRAIN}
fusion = json.loads(FUSION_PATH.read_text(encoding="utf-8"))
print("trained ensemble restored: OK")


## 16. 완전히 별도인 OOD 4개 pool × 625 준비

- Voice: 기본값은 In-the-Wild의 REAL/FAKE를 각각 625개씩 자동 다운로드합니다. SpeechFake test 폴더를 수동으로 준비하는 선택지도 남겨 둡니다.
- Real Music: Song Describer 625곡
- Fake Music: FakeMusicCaps 5개 생성기에서 각 125개

In-the-Wild 단일 ZIP은 약 8.16GB입니다. SpeechFake를 쓰려면 mode를 바꾸고 선택된 test 파일만 아래 두 Drive 폴더에 준비합니다.


In [ ]:
OOD_PROJECT = DRIVE_ROOT / "ood2500"
OOD_ARCHIVES = OOD_PROJECT / "archives"
OOD_RAW = Path("/content/deepvoice_ood_raw")
OOD_DATA = Path("/content/deepvoice_ood2500")
OOD_PROJECT.mkdir(parents=True, exist_ok=True)
OOD_ARCHIVES.mkdir(parents=True, exist_ok=True)
OOD_RAW.mkdir(parents=True, exist_ok=True)

OOD_POOL_SIZE = 625
OOD_SIZE = 2_500
VOICE_OOD_MODE = "in_the_wild_auto"  # 또는 speechfake_manual
SPEECHFAKE_REAL_DIR = DRIVE_ROOT / "ood_sources" / "speechfake" / "real_test"
SPEECHFAKE_FAKE_DIR = DRIVE_ROOT / "ood_sources" / "speechfake" / "fake_test"

OOD_URLS = {
    "in_the_wild": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip",
    "song_audio": "https://zenodo.org/api/records/10072001/files/audio.zip/content",
    "song_csv": "https://zenodo.org/api/records/10072001/files/song_describer.csv/content",
    "fake_music": "https://zenodo.org/api/records/15063698/files/FakeMusicCaps.zip/content",
    "musiccaps_csv": "https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv",
}
OOD_ARCHIVE_PATHS = {
    "in_the_wild": OOD_ARCHIVES / "release_in_the_wild.zip",
    "song_audio": OOD_ARCHIVES / "song_audio.zip",
    "song_csv": OOD_ARCHIVES / "song_describer.csv",
    "fake_music": OOD_ARCHIVES / "FakeMusicCaps.zip",
    "musiccaps_csv": OOD_ARCHIVES / "musiccaps-public.csv",
}


def download_resumable(url, destination):
    import requests
    temporary = destination.with_suffix(destination.suffix + ".part")
    existing = temporary.stat().st_size if temporary.exists() else 0
    headers = {"Range": f"bytes={existing}-"} if existing else {}
    mode = "ab" if existing else "wb"
    with requests.get(url, headers=headers, stream=True, timeout=(30, 600)) as response:
        if existing and response.status_code == 200:
            existing, mode = 0, "wb"
        response.raise_for_status()
        total = existing + int(response.headers.get("content-length", 0))
        with temporary.open(mode) as file, tqdm(total=total, initial=existing, unit="B", unit_scale=True) as bar:
            for chunk in response.iter_content(8 * 1024 * 1024):
                if chunk:
                    file.write(chunk); bar.update(len(chunk))
    temporary.replace(destination)


def safe_extract(archive_path, destination):
    destination = destination.resolve(); destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(member.filename)
        archive.extractall(destination)


def audio_files(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES)


DOWNLOAD_OOD = True
if DOWNLOAD_OOD:
    free_local_gib = shutil.disk_usage("/content").free / 2**30
    if free_local_gib < 40:
        raise RuntimeError(
            f"OOD 압축 해제와 2,500개 생성에 최소 40GiB 여유 공간을 권장합니다: {free_local_gib:.1f}GiB"
        )
    required = ["song_audio", "song_csv", "fake_music", "musiccaps_csv"]
    if VOICE_OOD_MODE == "in_the_wild_auto":
        required.insert(0, "in_the_wild")
    for key in required:
        if not OOD_ARCHIVE_PATHS[key].exists():
            download_resumable(OOD_URLS[key], OOD_ARCHIVE_PATHS[key])

for key, folder in {
    "in_the_wild": OOD_RAW / "in_the_wild",
    "song_audio": OOD_RAW / "song_describer",
    "fake_music": OOD_RAW / "fake_music_caps",
}.items():
    if key == "in_the_wild" and VOICE_OOD_MODE != "in_the_wild_auto":
        continue
    marker = folder / ".complete"
    if not marker.exists():
        safe_extract(OOD_ARCHIVE_PATHS[key], folder); marker.touch()
    print(key, len(audio_files(folder)))


In [ ]:
def normalize_label(value):
    value = str(value).strip().lower().replace("_", "-")
    if value in {"real", "bonafide", "bona-fide", "genuine", "0"}: return "real"
    if value in {"fake", "spoof", "deepfake", "1"}: return "fake"
    return None


def index_voice(root):
    files = audio_files(root); by_name = {p.name: p for p in files}; by_stem = {p.stem: p for p in files}
    rows = []
    for table_path in list(root.rglob("*.csv")) + list(root.rglob("*.tsv")):
        try: table = pd.read_csv(table_path, sep="\t" if table_path.suffix == ".tsv" else ",")
        except Exception: continue
        lower = {str(c).lower(): c for c in table.columns}
        label_col = next((lower[k] for k in ("label", "class", "target") if k in lower), None)
        file_col = next((lower[k] for k in ("file", "path", "filename", "audio") if k in lower), None)
        speaker_col = next((lower[k] for k in ("speaker", "speaker_id", "person") if k in lower), None)
        if not label_col or not file_col: continue
        for _, row in table.iterrows():
            label = normalize_label(row[label_col]); raw = Path(str(row[file_col]))
            path = by_name.get(raw.name) or by_stem.get(raw.stem)
            if label and path:
                rows.append({"path": str(path), "label": label, "group": str(row[speaker_col]) if speaker_col else path.parent.name})
        if rows: break
    return pd.DataFrame(rows).drop_duplicates("path")


def balanced_sample(frame, count, namespace):
    frame = frame.copy()
    frame["key"] = frame.path.map(lambda p: stable_int(f"ood|{CFG.seed}|{namespace}|{p}"))
    groups = [g.sort_values("key").to_dict("records") for _, g in frame.groupby("group")]
    chosen, cursor = [], 0
    while len(chosen) < count and any(cursor < len(group) for group in groups):
        for group in groups:
            if cursor < len(group) and len(chosen) < count: chosen.append(group[cursor])
        cursor += 1
    if len(chosen) != count: raise ValueError(f"{namespace}: {len(chosen)}/{count}")
    return pd.DataFrame(chosen).drop(columns="key")


if VOICE_OOD_MODE == "speechfake_manual":
    if not SPEECHFAKE_REAL_DIR.exists() or not SPEECHFAKE_FAKE_DIR.exists():
        raise FileNotFoundError(
            "SpeechFake test 파일을 "
            f"{SPEECHFAKE_REAL_DIR} 및 {SPEECHFAKE_FAKE_DIR}에 준비하거나 "
            "VOICE_OOD_MODE='in_the_wild_auto'로 변경하세요."
        )
    voice_index = pd.DataFrame(
        [{"path": str(p), "label": "real", "group": p.parent.name} for p in audio_files(SPEECHFAKE_REAL_DIR)] +
        [{"path": str(p), "label": "fake", "group": p.parent.name} for p in audio_files(SPEECHFAKE_FAKE_DIR)]
    )
else:
    voice_index = index_voice(OOD_RAW / "in_the_wild")
if voice_index.empty or "label" not in voice_index:
    raise RuntimeError("OOD voice metadata에서 real/fake 파일을 찾지 못했습니다.")
ood_real_voice = balanced_sample(voice_index[voice_index.label.eq("real")], OOD_POOL_SIZE, "real_voice")
ood_fake_voice = balanced_sample(voice_index[voice_index.label.eq("fake")], OOD_POOL_SIZE, "fake_voice")

song = pd.read_csv(OOD_ARCHIVE_PATHS["song_csv"]).sort_values("caption_id").groupby("track_id", as_index=False).agg({
    "path": "first", "artist_id": "first", "caption": " ".join,
})
song_files = audio_files(OOD_RAW / "song_describer")
song_lookup = {p.name.lower(): p for p in song_files}
song_rows = []
for _, row in song.iterrows():
    path = song_lookup.get(Path(str(row.path)).name.lower())
    if path: song_rows.append({"path": str(path), "group": str(row.artist_id), "caption": str(row.caption)})
ood_real_music = balanced_sample(pd.DataFrame(song_rows).drop_duplicates("path"), OOD_POOL_SIZE, "real_music")

musiccaps = pd.read_csv(OOD_ARCHIVE_PATHS["musiccaps_csv"])
caption_lookup = dict(zip(musiccaps.ytid.astype(str), musiccaps.caption.fillna("").astype(str)))
aliases = {"musicgen": "MusicGen", "musicldm": "MusicLDM", "audioldm2": "AudioLDM2", "stableaudioopen": "StableAudioOpen", "mustango": "Mustango"}
fake_rows = []
for path in audio_files(OOD_RAW / "fake_music_caps"):
    compact = re.sub(r"[^a-z0-9]", "", path.as_posix().lower())
    generator = next((name for token, name in aliases.items() if token in compact), None)
    if generator:
        fake_rows.append({"path": str(path), "group": generator, "generator": generator, "caption": caption_lookup.get(path.stem, "")})
fake_index = pd.DataFrame(fake_rows).drop_duplicates("path")
ood_fake_music = pd.concat([
    balanced_sample(fake_index[fake_index.generator.eq(generator)], 125, generator)
    for generator in sorted(set(aliases.values()))
], ignore_index=True)

ood_pools = {}
for name, frame in {"real_voice": ood_real_voice, "fake_voice": ood_fake_voice, "real_music": ood_real_music, "fake_music": ood_fake_music}.items():
    frame = frame.copy(); frame["pool"] = name; frame["source_id"] = [f"ood_{name}_{i:04d}" for i in range(len(frame))]
    if len(frame) != OOD_POOL_SIZE: raise RuntimeError((name, len(frame)))
    ood_pools[name] = frame
ood_source_manifest = pd.concat(ood_pools.values(), ignore_index=True)
if set(ood_source_manifest.path) & set(source_manifest.path):
    raise RuntimeError("training/OOD source overlap")
ood_source_manifest.to_csv(OOD_PROJECT / "ood_source_manifest.csv", index=False)
display(ood_source_manifest.groupby(["pool", "group"]).size().groupby(level=0).agg(["count", "min", "max", "sum"]))


## 17. DACON형 OOD DynamicMix 2,500개와 고정 정답

4~60초, 16 kHz, mono/stereo, WAV/FLAC/MP3/AAC/Opus, 12% 전화채널을 균형 있게 포함합니다. 이 test는 한 번 생성한 뒤 변경하지 않습니다.


In [ ]:
OOD_RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"], "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"], "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"], "fv_fm": ["fake_voice", "fake_music"],
}
VOICE_WORDS = re.compile(r"\b(vocal|voice|singer|singing|sung|lyrics|spoken|speech|whisper|rap|choir|chant)\b", re.I)


def np_audio(path):
    return decode_audio(path).numpy()


def crop_tile(audio, samples, rng):
    if len(audio) >= samples:
        start = int(rng.integers(0, len(audio) - samples + 1)); return audio[start:start + samples].copy()
    return np.tile(audio, math.ceil(samples / len(audio)))[:samples].copy()


def np_rms(audio): return float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)) + 1e-9)
def np_normalize(audio, db): return (audio * (10 ** (db / 20) / np_rms(audio))).astype(np.float32)


def mix_ood_components(components, rng):
    if len(components) == 1:
        return components[0].copy(), "single"
    layout = rng.choice(["overlap", "partial", "sequential"], p=[.50, .25, .25])
    first, second = components
    if layout == "overlap":
        return (first + second).astype(np.float32), layout
    if layout == "partial":
        length = len(first)
        start = int(rng.integers(0, max(1, int(length * .45))))
        stop = int(rng.integers(max(start + 1, int(length * .60)), length + 1))
        gain = np.zeros(length, np.float32)
        fade = min(int(.05 * 16000), max(1, (stop - start) // 4))
        gain[start:stop] = 1
        gain[start:start + fade] = np.linspace(0, 1, fade, dtype=np.float32)
        gain[stop - fade:stop] = np.linspace(1, 0, fade, dtype=np.float32)
        return (first + second * gain).astype(np.float32), layout
    length = len(first)
    boundary = int(rng.integers(int(length * .35), max(int(length * .35) + 1, int(length * .65))))
    fade = min(int(.10 * 16000), boundary, length - boundary)
    first_gain = np.ones(length, np.float32); second_gain = np.zeros(length, np.float32)
    first_gain[boundary:] = 0; second_gain[boundary:] = 1
    if fade:
        first_gain[boundary - fade:boundary + fade] = np.linspace(1, 0, 2 * fade, dtype=np.float32)
        second_gain[boundary - fade:boundary + fade] = np.linspace(0, 1, 2 * fade, dtype=np.float32)
    return (first * first_gain + second * second_gain).astype(np.float32), layout


def phone_filter(audio):
    down = librosa.resample(audio, orig_sr=16000, target_sr=8000, res_type="soxr_hq")
    tensor = torch.from_numpy(np.asarray(down, np.float32)).clamp(-1, 1)
    tensor = torchaudio.functional.highpass_biquad(tensor, 8000, 300)
    tensor = torchaudio.functional.lowpass_biquad(tensor, 8000, 3400)
    tensor = torchaudio.functional.mu_law_decoding(torchaudio.functional.mu_law_encoding(tensor, 256), 256)
    return librosa.resample(tensor.numpy(), orig_sr=8000, target_sr=16000, res_type="soxr_hq").astype(np.float32)


CODECS = [("wav", None), ("flac", None), ("mp3", "64k"), ("mp3", "96k"), ("mp3", "128k"),
          ("aac", "64k"), ("aac", "96k"), ("opus", "48k"), ("opus", "64k")]


def encode_ood(audio, destination, codec, bitrate):
    temporary = destination.with_suffix(".input.wav"); sf.write(temporary, audio, 16000, subtype="PCM_16")
    args = {"wav": ["-c:a", "pcm_s16le"], "flac": ["-c:a", "flac"],
            "mp3": ["-c:a", "libmp3lame", "-b:a", bitrate], "aac": ["-c:a", "aac", "-b:a", bitrate],
            "opus": ["-c:a", "libopus", "-b:a", bitrate]}[codec]
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(temporary), "-ar", "16000", *args, str(destination)], check=True)
    temporary.unlink()


BUILD_OOD = True
if BUILD_OOD:
    if OOD_DATA.exists(): shutil.rmtree(OOD_DATA)
    test_dir = OOD_DATA / "data" / "test"; test_dir.mkdir(parents=True)
    rng = np.random.default_rng(CFG.seed + 9001)
    recipe_types = [name for index, name in enumerate(OOD_RECIPE_COMPONENTS) for _ in range(313 if index < 4 else 312)]
    random.Random(CFG.seed + 9001).shuffle(recipe_types)
    duration_values = rng.choice([4, 5, 6, 8, 10, 12, 15, 20, 30, 45, 60], OOD_SIZE,
                                 p=np.array([8, 8, 10, 12, 14, 12, 10, 8, 5, 2, 1]) / 90)
    records = {name: frame.sample(frac=1, random_state=CFG.seed).to_dict("records") for name, frame in ood_pools.items()}
    cursors = {name: 0 for name in records}; manifest_rows = []; truth_rows = []
    for index, (recipe, seconds) in enumerate(tqdm(zip(recipe_types, duration_values), total=OOD_SIZE)):
        identifier = f"OOD_{index:05d}"; samples = int(seconds * 16000); components = []; selected_rows = []
        for pool in OOD_RECIPE_COMPONENTS[recipe]:
            row = records[pool][cursors[pool] % len(records[pool])]; cursors[pool] += 1
            components.append(np_normalize(crop_tile(np_audio(row["path"]), samples, rng), float(rng.uniform(-25, -19))))
            selected_rows.append(row)
        mixed, layout = mix_ood_components(components, rng)
        telephone = bool(rng.random() < .12)
        if telephone: mixed = crop_tile(phone_filter(mixed), samples, rng)
        mixed = np_normalize(mixed - mixed.mean(), float(rng.uniform(-24, -18)))
        mixed = np.clip(mixed / max(np.max(np.abs(mixed)), 1e-8) * float(rng.uniform(.72, .96)), -1, 1)
        stereo = bool(rng.random() < .30)
        if stereo: mixed = np.stack([mixed, np.roll(mixed, int(rng.integers(2, 32))) * .94], axis=1)
        codec, bitrate = CODECS[index % len(CODECS)]; extension = "m4a" if codec == "aac" else codec
        encode_ood(mixed, test_dir / f"{identifier}.{extension}", codec, bitrate)

        def has_voice(row): return row["pool"].endswith("voice") or bool(VOICE_WORDS.search(str(row.get("caption", ""))))
        vp = int(any(has_voice(row) for row in selected_rows)); mp = int(any(row["pool"].endswith("music") for row in selected_rows))
        vf = int(any(row["pool"] == "fake_voice" or (row["pool"] == "fake_music" and has_voice(row)) for row in selected_rows))
        mf = int(any(row["pool"] == "fake_music" for row in selected_rows)); ff = max(vf, mf)
        manifest_rows.append({"ID": identifier, "recipe_type": recipe, "duration": seconds, "codec": codec,
                              "bitrate": bitrate or "lossless", "channels": 2 if stereo else 1, "telephone": telephone,
                              "layout": layout,
                              "sources": "|".join(row["source_id"] for row in selected_rows)})
        truth_rows.append({"ID": identifier, "FILE_FAKE": ff, "VOICE_FAKE": vf, "MUSIC_FAKE": mf, "VOICE_PRESENT": vp, "MUSIC_PRESENT": mp})
    sample = pd.DataFrame({"ID": [row["ID"] for row in manifest_rows]})
    for column in DACON_PROBABILITY_COLUMNS: sample[column] = 0.0
    sample.to_csv(OOD_DATA / "data" / "sample_submission.csv", index=False)
    pd.DataFrame(manifest_rows).to_csv(OOD_DATA / "manifest.csv", index=False)
    pd.DataFrame(truth_rows).to_csv(OOD_DATA / "ground_truth.csv", index=False)
    print("OOD files:", len(audio_files(test_dir)))


## 18. 독립 OOD 2,500개 최종 평가

중간 저장을 지원합니다. OOD 점수는 모델·fusion weight 변경에 사용하지 않습니다.


In [ ]:
RUN_OOD_TEST = True
OOD_PREDICTIONS_PATH = OOD_PROJECT / "predictions.csv"

if RUN_OOD_TEST:
    sample = pd.read_csv(OOD_DATA / "data" / "sample_submission.csv")
    existing = pd.read_csv(OOD_PREDICTIONS_PATH) if OOD_PREDICTIONS_PATH.exists() else pd.DataFrame(columns=["ID", *DACON_PROBABILITY_COLUMNS])
    completed = set(existing.ID.astype(str)); rows = existing.to_dict("records")
    lookup = {p.stem: p for p in audio_files(OOD_DATA / "data" / "test")}
    for index, identifier in enumerate(tqdm(sample.ID.astype(str), desc="OOD ensemble"), 1):
        if identifier in completed: continue
        probabilities = predict_file_ensemble(decode_audio(lookup[identifier]), trained_models, fusion)
        rows.append({"ID": identifier, **dict(zip(DACON_PROBABILITY_COLUMNS, map(float, probabilities)))})
        if len(rows) % 50 == 0: pd.DataFrame(rows).to_csv(OOD_PREDICTIONS_PATH, index=False)
    prediction = pd.DataFrame(rows).drop_duplicates("ID", keep="last").set_index("ID").loc[sample.ID.astype(str)].reset_index()
    prediction.to_csv(OOD_PREDICTIONS_PATH, index=False)
    truth = pd.read_csv(OOD_DATA / "ground_truth.csv")
    report = dacon_official_score(truth[DACON_TRUTH_COLUMNS], prediction[DACON_PROBABILITY_COLUMNS])
    (OOD_PROJECT / "official_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    display(pd.DataFrame([report]).T.rename(columns={0: "value"}))
    print("주의: music 내부 voice 여부는 caption keyword 기반 보조 label입니다.")


## 19. L4 기준 50분 추론 time gate

공식 1,200개 test를 가정해 OOD 파일 80개를 실제 ensemble로 처리하고, 파일당 시간에 15% 안전계수와 1.5분 로딩 여유를 더합니다. L4가 아닌 GPU의 속도로 통과시키면 안 되므로 기본값은 L4를 강제합니다.


In [ ]:
RUN_L4_TIME_GATE = True
REQUIRE_L4_FOR_GATE = True
OFFICIAL_TEST_FILES = 1_200
TIME_GATE_TARGET_MINUTES = 50.0
TIME_GATE_SAMPLE_FILES = 80
TIME_GATE_PATH = RUN_ROOT / "l4_time_gate.json"
TIME_GATE_RESULT = {"passed": False, "reason": "not run"}

if RUN_L4_TIME_GATE:
    if DEVICE.type != "cuda":
        raise RuntimeError("L4 time gate에는 CUDA GPU가 필요합니다.")
    gpu_name = torch.cuda.get_device_name(0)
    if REQUIRE_L4_FOR_GATE and "L4" not in gpu_name.upper():
        raise RuntimeError(f"L4에서 다시 실행해야 합니다. current GPU={gpu_name}")
    manifest = pd.read_csv(OOD_DATA / "manifest.csv").sort_values("duration", ascending=False)
    chosen_ids = manifest.ID.astype(str).head(TIME_GATE_SAMPLE_FILES).tolist()
    lookup = {p.stem: p for p in audio_files(OOD_DATA / "data" / "test")}
    missing = [identifier for identifier in chosen_ids if identifier not in lookup]
    if missing:
        raise FileNotFoundError(missing[:5])
    # 첫 호출의 CUDA kernel 초기화 비용도 측정에 포함한다.
    torch.cuda.synchronize(); started = time.perf_counter()
    for identifier in tqdm(chosen_ids, desc="L4 time gate"):
        predict_file_ensemble(decode_audio(lookup[identifier]), trained_models, fusion)
    torch.cuda.synchronize(); elapsed = time.perf_counter() - started
    projected_minutes = 1.5 + elapsed / len(chosen_ids) * OFFICIAL_TEST_FILES * 1.15 / 60
    TIME_GATE_RESULT = {
        "gpu": gpu_name, "sample_files": len(chosen_ids), "sample_seconds": elapsed,
        "projected_1200_minutes": projected_minutes,
        "target_minutes": TIME_GATE_TARGET_MINUTES,
        "maximum_segments": fusion["aggregation"]["maximum_segments"],
        "passed": bool(projected_minutes <= TIME_GATE_TARGET_MINUTES),
    }
    TIME_GATE_PATH.write_text(json.dumps(TIME_GATE_RESULT, indent=2), encoding="utf-8")
    print(json.dumps(TIME_GATE_RESULT, indent=2))
    if not TIME_GATE_RESULT["passed"]:
        raise RuntimeError("50분 time gate 실패: maximum_segments 또는 ensemble을 줄여야 합니다.")


## 20. 오프라인 DACON `script.py`

학습된 세 checkpoint, AASIST 코드, XLS-R config, validation에서 고정한 fusion을 포함합니다. 평가 파일은 서로 독립적으로 처리하며 다른 test 파일의 통계로 보정하지 않습니다.


In [ ]:
INFERENCE_SCRIPT = r'''from __future__ import annotations

import json, math, subprocess, sys
from pathlib import Path
import librosa, numpy as np, pandas as pd, soundfile as sf, torch, torch.nn as nn, torch.nn.functional as F, torchaudio
from transformers import AutoConfig, AutoModel

ROOT = Path(__file__).resolve().parent
MODEL_DIR = ROOT / "model"
DATA_DIR = next((ROOT / name for name in ("data", "open") if (ROOT / name).exists()), ROOT / "data")
TEST_DIR = DATA_DIR / "test"
OUTPUT = ROOT / "output" / "submission.csv"
SR, CLIP = 16000, 64600
COLUMNS = ["FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB", "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB"]
SUFFIXES = {".wav", ".flac", ".mp3", ".m4a", ".aac", ".ogg", ".opus", ".wma", ".amr"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PresenceLogMel(nn.Module):
    def __init__(self, dropout=.20):
        super().__init__(); self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=1024, win_length=400, hop_length=160, n_mels=96, f_min=20, f_max=7600)
        self.encoder = nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.BatchNorm2d(32),nn.SiLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.SiLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.SiLU(),nn.MaxPool2d(2),nn.Conv2d(128,192,3,padding=1),nn.BatchNorm2d(192),nn.SiLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(192,2))
    def forward(self,audio):
        x=torch.log(self.mel(audio).clamp_min(1e-6)); x=(x-x.mean((-2,-1),keepdim=True))/(x.std((-2,-1),keepdim=True)+1e-5); return self.head(self.encoder(x.unsqueeze(1)))

class AttentionBlock(nn.Module):
    def __init__(self,dim,dropout=.1):
        super().__init__(); self.norm1=nn.LayerNorm(dim); self.attention=nn.MultiheadAttention(dim,4,dropout=dropout,batch_first=True); self.norm2=nn.LayerNorm(dim); self.ff=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim))
    def forward(self,x):
        z=self.norm1(x); x=x+self.attention(z,z,z,need_weights=False)[0]; return x+self.ff(self.norm2(x))

class XLSRDualGraphFake3(nn.Module):
    def __init__(self,dropout=.2,dim=128):
        super().__init__(); config=AutoConfig.from_pretrained(MODEL_DIR/"xlsr_config",local_files_only=True); self.ssl=AutoModel.from_config(config); hidden=self.ssl.config.hidden_size
        self.projection=nn.Linear(hidden,dim); self.feature_projection=nn.Linear(hidden,dim); self.time_graph=nn.Sequential(AttentionBlock(dim),AttentionBlock(dim)); self.feature_graph=nn.Sequential(AttentionBlock(dim),AttentionBlock(dim)); self.head=nn.Sequential(nn.LayerNorm(dim*4),nn.Dropout(dropout),nn.Linear(dim*4,256),nn.GELU(),nn.Linear(256,3))
    def forward(self,audio):
        audio=(audio-audio.mean(1,keepdim=True))/(audio.std(1,keepdim=True)+1e-5); raw=self.ssl(audio).last_hidden_state; hidden=self.projection(raw); t=F.adaptive_avg_pool1d(hidden.transpose(1,2),64).transpose(1,2); f=self.feature_projection(F.adaptive_max_pool1d(raw.transpose(1,2),8).transpose(1,2)); t=self.time_graph(t); f=self.feature_graph(f); return self.head(torch.cat([t.mean(1),t.amax(1),f.mean(1),f.amax(1)],-1))

def build_aasist():
    sys.path.insert(0,str(MODEL_DIR/"aasist")); config=json.loads((MODEL_DIR/"aasist"/"config"/"AASIST.conf").read_text())["model_config"]
    from models.AASIST import Model
    class Wrapper(nn.Module):
        def __init__(self): super().__init__(); self.net=Model(config); self.net.out_layer=nn.Linear(self.net.out_layer.in_features,3)
        def forward(self,x): return self.net(x,Freq_aug=False)[1]
    return Wrapper()

def restore(name, model):
    state=torch.load(MODEL_DIR/name/"best.pt",map_location="cpu",weights_only=False); model.load_state_dict(state["model_state"],strict=True); return model.to(DEVICE).eval()

def read_audio(path):
    try: data,sr=sf.read(path,dtype="float32",always_2d=True); audio=torch.from_numpy(data.mean(1))
    except Exception:
        try: data,sr=librosa.load(path,sr=None,mono=True); audio=torch.from_numpy(np.asarray(data,np.float32))
        except Exception:
            result=subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-i",str(path),"-ac","1","-ar","16000","-f","f32le","pipe:1"],stdout=subprocess.PIPE,check=True,timeout=120); audio=torch.from_numpy(np.frombuffer(result.stdout,dtype="<f4").copy()); sr=16000
    if sr!=SR: audio=torchaudio.functional.resample(audio,sr,SR)
    return audio.float().nan_to_num().clamp(-1,1)

def segments(audio,maximum):
    if audio.numel()<CLIP: audio=audio.repeat(math.ceil(CLIP/max(1,audio.numel())))
    starts=list(range(0,audio.numel()-CLIP+1,CLIP)); last=audio.numel()-CLIP
    if not starts or starts[-1]!=last: starts.append(last)
    if len(starts)>maximum: starts=[starts[i] for i in sorted(set(np.linspace(0,len(starts)-1,maximum).round().astype(int)))]
    result=[]
    for start in starts:
        x=audio[start:start+CLIP]; x=x-x.mean(); x=x/x.abs().max().clamp_min(1e-8)*.82; result.append(x.clamp(-1,1))
    return torch.stack(result)

def infer(model,x,batch):
    out=[]
    with torch.inference_mode():
        for start in range(0,len(x),batch):
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=="cuda"): logits=model(x[start:start+batch].to(DEVICE))
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def blend(a,b,w):
    e=1e-5; a=np.clip(a,e,1-e); b=np.clip(b,e,1-e); z=w*np.log(a/(1-a))+(1-w)*np.log(b/(1-b)); return 1/(1+np.exp(-z))
def topk(x,f): x=np.asarray(x); k=max(1,math.ceil(len(x)*f)); return float(np.sort(x)[-k:].mean())
def gated(fake,presence,f):
    k=max(1,math.ceil(len(fake)*max(.5,f))); idx=np.argsort(presence)[-k:]; return topk(np.asarray(fake)[idx],f)

def predict(path,models,fusion):
    x=segments(read_audio(path),fusion["aggregation"]["maximum_segments"]); p=infer(models["presence"],x,64); a=infer(models["aasist"],x,32); d=infer(models["xlsr"],x,4); f=np.zeros_like(a)
    for i,name in enumerate(("file_fake","voice_fake","music_fake")): f[:,i]=blend(a[:,i],d[:,i],fusion["weights"][name+"_aasist_weight"])
    vp=topk(p[:,0],.3); mp=topk(p[:,1],.3); vf=gated(f[:,1],p[:,0],.3); mf=gated(f[:,2],p[:,1],.3); direct=topk(f[:,0],.3); coherent=1-(1-vp*vf)*(1-mp*mf); ff=fusion["file_direct_weight"]*direct+(1-fusion["file_direct_weight"])*coherent
    return np.clip([ff,vf,mf,vp,mp],0,1)

def main():
    sample=pd.read_csv(DATA_DIR/"sample_submission.csv"); required=["ID",*COLUMNS]
    if list(sample.columns)!=required: raise ValueError(list(sample.columns))
    files=sorted(p for p in TEST_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUFFIXES); lookup={p.stem:p for p in files}; lookup.update({p.name:p for p in files})
    models={"presence":restore("presence",PresenceLogMel()),"aasist":restore("aasist_fake3",build_aasist()),"xlsr":restore("xlsr_dualgraph_fake3",XLSRDualGraphFake3())}; fusion=json.loads((MODEL_DIR/"fusion.json").read_text())
    rows=[]
    for identifier in sample.ID.astype(str):
        path=lookup.get(identifier)
        if path is None: raise FileNotFoundError(identifier)
        rows.append([identifier,*predict(path,models,fusion)])
    result=pd.DataFrame(rows,columns=required); values=result[COLUMNS].to_numpy(float)
    if not np.isfinite(values).all() or not ((values>=0)&(values<=1)).all(): raise ValueError("invalid probabilities")
    OUTPUT.parent.mkdir(parents=True,exist_ok=True); result.to_csv(OUTPUT,index=False,encoding="utf-8"); print("saved",OUTPUT,len(result))

if __name__=="__main__": main()
'''

ast.parse(INFERENCE_SCRIPT)
print("offline script AST: OK")


## 21. `submit.zip` 생성·복원·용량 검증


In [ ]:
from transformers import AutoConfig

BUILD_SUBMIT = True
SUBMIT_ZIP = DRIVE_ROOT / "submit.zip"
if BUILD_SUBMIT:
    if not RUN_L4_TIME_GATE or not TIME_GATE_RESULT.get("passed", False):
        raise RuntimeError("L4 50분 time gate 통과 후 submit.zip을 생성하세요.")
    stage = Path("/content/deepvoice_v3_submit")
    if stage.exists(): shutil.rmtree(stage)
    model_dir = stage / "model"; model_dir.mkdir(parents=True)
    for name in BRANCHES_TO_TRAIN:
        source = RUN_ROOT / name / "best.pt"
        if not source.exists(): raise FileNotFoundError(source)
        target = model_dir / name; target.mkdir(); shutil.copy2(source, target / "best.pt")
    shutil.copy2(FUSION_PATH, model_dir / "fusion.json")
    shutil.copytree(REPO_ROOT / "aasist" / "models", model_dir / "aasist" / "models")
    shutil.copytree(REPO_ROOT / "aasist" / "config", model_dir / "aasist" / "config")
    AutoConfig.from_pretrained(XLSR_MODEL, revision=XLSR_REVISION).save_pretrained(model_dir / "xlsr_config")
    (stage / "script.py").write_text(INFERENCE_SCRIPT, encoding="utf-8")
    (stage / "requirements.txt").write_text("# Uses DACON preinstalled packages.\n", encoding="utf-8")
    import py_compile
    py_compile.compile(str(stage / "script.py"), doraise=True)

    # CPU에서 세 checkpoint의 key/shape를 현재 클래스에 strict restore하여 packaging 오류를 먼저 잡는다.
    for name in BRANCHES_TO_TRAIN:
        state = torch.load(RUN_ROOT / name / "best.pt", map_location="cpu", weights_only=False)
        model = build_branch(name); model.load_state_dict(state["model_state"], strict=True); del model
    gc.collect()

    with zipfile.ZipFile(SUBMIT_ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True) as archive:
        for path in sorted(stage.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(stage).as_posix())
    with zipfile.ZipFile(SUBMIT_ZIP) as archive:
        if archive.testzip(): raise RuntimeError("corrupt zip")
        names = archive.namelist(); top = {name.split("/",1)[0] for name in names}
        unpacked = sum(info.file_size for info in archive.infolist())
    compressed = SUBMIT_ZIP.stat().st_size
    if top != {"model", "script.py", "requirements.txt"}: raise RuntimeError(top)
    if compressed >= 10*2**30 or unpacked >= 32*2**30: raise RuntimeError((compressed/2**30, unpacked/2**30))
    print("created:", SUBMIT_ZIP)
    print({"zip_GiB": compressed/2**30, "unpacked_GiB": unpacked/2**30})


## 최종 실행 체크리스트

1. Colab L4 GPU와 Google Drive 여유 공간을 확인합니다.
2. Kaggle credential, FMA, SONICS 다운로드 셀을 실행합니다.
3. 각 training pool 6,250개, train 22,500 recipe, validation 2,500 recipe를 확인합니다.
4. Presence → AASIST Fake3 → XLS-R Dual-Graph Fake3 순서로 학습합니다. XLS-R 학습은 가장 오래 걸립니다.
5. clean/stress validation에서 fusion weight를 한 번 고정합니다.
6. 기본 In-the-Wild와 OOD music을 자동 다운로드해 OOD 2,500개를 한 번만 평가합니다.
7. L4 50분 time gate, `submit.zip` 크기 및 strict 복원 검사를 통과한 뒤 DACON에 제출합니다.

`official_metrics.json`의 OOD 점수는 리더보드 점수를 보장하지 않으며, 특히 음악 내부 Voice Presence는 caption keyword 기반 보조 정답이라는 한계가 있습니다.
